<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/Classical_Text_Vectorization_One_Hot_Encoding_to_Latent_Semantic_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Классические методы векторизации текста: от one-hot encoding до латентного семантического анализа

## Введение

Любая задача обработки естественного языка (NLP) — классификация документов, кластеризация, информационный поиск, машинный перевод — требует перевода текста в числовую форму, пригодную для математических моделей и алгоритмов машинного обучения. Выбор способа такого представления напрямую определяет, какие закономерности языка сможет уловить модель, насколько компактным и вычислительно эффективным будет векторное пространство и как оно поведёт себя при работе с шумом и редкими словами.

В этой серии лекций рассматривается эволюция **классических** методов векторизации текста — от простейшего one-hot encoding до вероятностной тематической модели латентного размещения Дирихле (LDA). Каждый метод излагается по единой схеме: математическая формализация, наглядный числовой пример на фиксированном учебном корпусе из трёх документов и критический анализ ограничений. Такой подход позволяет наглядно проследить, как усложнение моделей приводит к более содержательным и компактным представлениям текста.

Изучение классических подходов необходимо для понимания принципов работы современных нейросетевых эмбеддингов (Word2Vec, BERT и др.), которые будут рассмотрены в следующих сериях.

---

## Часть 1. Введение в методы векторизации текста и one-hot encoding

### 1.1 Зачем нужна векторизация текста

Текст в естественном виде — последовательность символов или слов — не может быть непосредственно передан в алгоритмы машинного обучения, которые оперируют числами. Поэтому первым шагом любого NLP-проекта является **векторизация** — отображение текстовых единиц (слов, предложений, документов) в числовые векторы, максимально сохраняющие содержательную информацию исходного текста.

Понятие векторизации охватывает широкий спектр методов, различающихся по сложности, вычислительной эффективности и способности улавливать семантические отношения. Историческое развитие этих методов можно представить как последовательное преодоление ограничений предыдущих подходов:

1. **One-hot encoding** — самый простой способ, который лишь уникально идентифицирует слово, полностью игнорируя его смысл.
2. **Частотные модели документов (Bag of Words, TF-IDF)** — учитывают статистическую значимость терминов, но не улавливают смысловую близость слов.
3. **Латентные методы (LSA, тематические модели)** — выявляют скрытые факторы и позволяют словам и документам находиться в общем непрерывном пространстве.
4. **Нейросетевые эмбеддинги (Word2Vec, BERT)** — дают плотные семантически насыщенные представления, обучаемые на огромных корпусах.

В этой части мы детально разберём первый, базовый способ представления слова — one-hot encoding. Несмотря на кажущуюся примитивность, он важен для понимания того, почему потребовались более сложные подходы и какие принципиальные проблемы возникают при работе с естественным языком.

### 1.2 Понятие словаря и индексного пространства

Пусть имеется некоторая коллекция текстов — **корпус**. Корпус может состоять из одного или нескольких документов, а документы — из последовательности слов. Под словом понимается лексическая единица после предварительной обработки: приведения к нижнему регистру, удаления знаков препинания, лемматизации или стемминга. Для простоты будем считать, что слова уже нормализованы и представляют собой минимальные смысловые единицы, разделённые пробелами.

Из всего корпуса извлекается множество уникальных слов — **словарь** $V$. Его размер $|V|$ (или $N$) может варьироваться от нескольких десятков в учебных примерах до нескольких миллионов в реальных корпусах. Каждому слову $w \in V$ присваивается уникальный целочисленный индекс $i \in \{1, 2, \ldots, N\}$. Такое соответствие задаётся биективной функцией индексирования:
$$\text{id}: V \to \{1, 2, \ldots, N\}.$$
Порядок присвоения индексов может быть произвольным, но на практике часто используют сортировку по алфавиту или по убыванию частоты встречаемости.

Когда все слова корпуса заменяются их индексами, текст превращается в последовательность целых чисел. Однако такое представление неудобно для большинства алгоритмов машинного обучения, поскольку числа вводят искусственный порядок, не отражающий семантических отношений. Например, если слову «кошка» присвоен индекс 1, а слову «собака» индекс 2, то разность индексов может быть ошибочно истолкована как мера близости. Поэтому возникает необходимость в векторном представлении, в котором каждое слово было бы точкой в некотором векторном пространстве. Простейшим из таких представлений является one-hot encoding.



## 1.3 Определение one-hot encoding

One-hot encoding, или унитарное кодирование, сопоставляет каждому слову $w_i$ из словаря $V$ вектор $e_i$ размерности $N = |V|$. Все компоненты этого вектора равны нулю, за исключением одной — той, которая соответствует индексу данного слова. Если зафиксировать порядок слов в словаре, например, $V = \{w_1, w_2, \ldots, w_N\}$, то вектор для слова $w_i$ имеет вид:

$$
e_i = \left( 0, \; 0, \; \ldots, \; 0, \; \underbrace{1}_{i\text{-я позиция}}, \; 0, \; \ldots, \; 0 \right)^\top ,
$$

где символ $\top$ обозначает транспонирование, то есть вектор записан как столбец, но для удобства часто изображается строкой.

Формально:

$$
(e_i)_j =
\begin{cases}
1, & \text{если } j = i, \\
0, & \text{если } j \neq i.
\end{cases}
$$

Такой вектор содержит ровно одну единицу и $N-1$ нулей. Матрица, составленная из всех one-hot векторов словаря, является единичной матрицей размером $N \times N$:

$$
E = \begin{pmatrix}
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & 1
\end{pmatrix} = I_N .
$$

Это свойство проистекает из ортонормированности системы векторов: для любых $i, j$ скалярное произведение

$$
e_i^\top e_j =
\begin{cases}
1, & \text{если } i = j, \\
0, & \text{если } i \neq j.
\end{cases}
$$

Данное равенство показывает, что one-hot векторы попарно ортогональны и имеют единичную норму. Именно эта ортогональность становится ключевым источником как достоинств (простота, однозначность), так и фатальных недостатков (отсутствие семантической близости), о которых пойдёт речь ниже.

## 1.4 Пример one-hot encoding на маленьком корпусе

Рассмотрим иллюстративный корпус, состоящий из трёх документов:

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».

После удаления знаков препинания и приведения к нормальной форме получаем следующий набор уникальных слов (словарь):

$$
V = \{ \text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване} \}.
$$

Размер словаря $N = 8$. Присвоим каждому слову индекс в порядке перечисления: кошка — 1, сидит — 2, на — 3, окне — 4, собака — 5, крыльце — 6, спит — 7, диване — 8. Тогда one-hot векторы для каждого слова будут иметь длину 8.

Для слова «кошка» (индекс 1) вектор равен:

$$
e_{\text{кошка}} = (1, 0, 0, 0, 0, 0, 0, 0)^\top .
$$

Для слова «сидит» (индекс 2):

$$
e_{\text{сидит}} = (0, 1, 0, 0, 0, 0, 0, 0)^\top .
$$

Аналогично,

$$
e_{\text{на}} = (0, 0, 1, 0, 0, 0, 0, 0)^\top ,
$$

$$
e_{\text{окне}} = (0, 0, 0, 1, 0, 0, 0, 0)^\top ,
$$

$$
e_{\text{собака}} = (0, 0, 0, 0, 1, 0, 0, 0)^\top ,
$$

$$
e_{\text{крыльце}} = (0, 0, 0, 0, 0, 1, 0, 0)^\top ,
$$

$$
e_{\text{спит}} = (0, 0, 0, 0, 0, 0, 1, 0)^\top ,
$$

$$
e_{\text{диване}} = (0, 0, 0, 0, 0, 0, 0, 1)^\top .
$$

Если мы захотим представить целый документ как последовательность one-hot векторов, то документ 1 будет записан как список из четырёх векторов:

$$
[ e_{\text{кошка}}, e_{\text{сидит}}, e_{\text{на}}, e_{\text{окне}} ].
$$

Такое представление сохраняет порядок слов, но каждый вектор имеет длину $N$, равную размеру словаря. Для нашего игрушечного корпуса $N=8$, что не вызывает проблем, однако при реальных объёмах словарь достигает сотен тысяч и миллионов, и хранение последовательностей таких векторов становится крайне неэффективным, как будет показано в следующем разделе.

## 1.5 Вычислительная и семантическая неэффективность one-hot encoding

### 1.5.1 Проблема высокой размерности и разреженности

Главным недостатком one-hot encoding является колоссальная размерность получаемых векторов. Если словарь содержит $N$ слов, то каждый вектор имеет длину $N$, а поскольку реальные $N$ могут составлять от $10^5$ до $10^7$, хранение плотного вектора для каждого вхождения слова становится непрактичным. Более того, подавляющее большинство компонентов вектора равны нулю. Доля ненулевых элементов равна $1/N$, что при $N = 10^6$ составляет $10^{-6}$, то есть 0,0001 %. Такие векторы называют разреженными (sparse). Хотя существуют специализированные форматы хранения разреженных матриц и векторов, они не устраняют фундаментальной проблемы: модель, работающая с такими представлениями, вынуждена оперировать пространством, в котором каждое слово изолировано от остальных.

Чтобы наглядно продемонстрировать масштаб проблемы, проведём мысленный эксперимент. Представьте небольшую библиотеку из 15 000 книг. Каждая книга содержит около 300 страниц, на каждой странице в среднем 35 строк, а в каждой строке — 20 слов. Тогда общее количество словоупотреблений во всей библиотеке равно:

$$
T = 15\,000 \times 300 \times 35 \times 20 = 3\,150\,000\,000 .
$$

Если словарь уникальных слов после нормализации составляет $N = 1\,000\,000$ (что реалистично для такого объёма), то каждый one-hot вектор займёт $N$ чисел. При использовании 32-битных чисел с плавающей точкой (float32), каждый вектор потребует $4 \times 10^6$ байт = 4 МБ. Для хранения векторов для всех вхождений потребуется $T \times 4 \times 10^6$ байт, то есть $12{,}6 \times 10^{15}$ байт, или 12,6 петабайт. Это на несколько порядков превышает объёмы, доступные даже крупным дата-центрам. Даже если использовать разреженное представление, храня только индекс ненулевого элемента, объём сократится до $T \times 4$ байт ≈ 12,6 ГБ, что уже приемлемо, но тогда мы фактически отказываемся от one-hot векторов в пользу простых индексов, и никакой дополнительной информации не получаем.

Таким образом, разреженное хранение one-hot векторов эквивалентно возврату к простой индексации слов, не дающей никакой дополнительной информации о их значениях. Поэтому one-hot encoding используется лишь как промежуточный технический приём, а не как самостоятельное представление для семантических задач.

Проблема не только в памяти, но и в вычислительной сложности. Матричные операции с такими векторами, например умножение матрицы весов, требуют либо огромных ресурсов, либо специальных разреженных процедур, которые всё равно ограничены. Но даже если бы память и вычисления были бесплатными, остаётся ещё более фундаментальная проблема — отсутствие семантической близости.

### 1.5.2 Отсутствие семантической близости

Из определения one-hot векторов следует, что скалярное произведение любых двух различных векторов равно нулю:

$$
e_i^\top e_j = 0 \quad \text{при } i \neq j .
$$

Это означает, что в данном векторном пространстве все слова одинаково далеки друг от друга с точки зрения евклидова расстояния или косинусной меры. Действительно, евклидово расстояние между $e_i$ и $e_j$ равно $\sqrt{2}$ для всех $i \neq j$, а косинусная близость равна 0. Следовательно, слова «кошка» и «собака» оказываются не ближе, чем «кошка» и «диван» или «кошка» и «на». Семантическая информация, присущая языку, полностью теряется. Дистрибутивная гипотеза, согласно которой слова со схожими значениями встречаются в схожих контекстах, в one-hot представлении не находит никакого отражения.

Более того, one-hot векторы не позволяют вводить понятие частичного сходства: невозможно сказать, что «кошка» и «котёнок» похожи, а «кошка» и «автомобиль» — нет. Каждое слово становится изолированной точкой на сфере в $N$-мерном пространстве, и никакая линейная или нелинейная модель, обученная на таких векторах без дополнительной информации, не сможет выявить семантические закономерности. Именно поэтому one-hot encoding используется в основном как промежуточный технический приём для кодирования категориальных признаков или как вход для embedding-слоёв нейронных сетей, где он немедленно преобразуется в плотные низкоразмерные векторы.

### 1.5.3 Проблема неограниченности словаря

Ещё один недостаток one-hot encoding связан с тем, что векторное пространство жёстко привязано к словарю обучающего корпуса. Если после обучения модели встречается слово, которого не было в корпусе (out-of-vocabulary, OOV), то его невозможно представить в том же векторном пространстве без перестройки всего представления. Приходится либо добавлять новое измерение и переобучать модель, либо заменять такие слова специальным токеном (например, `<unk>`), что приводит к потере информации. В реальных задачах OOV-слова возникают постоянно из-за новых терминов, имён, опечаток и морфологических вариаций, поэтому подобная негибкость является серьёзным ограничением.

Таким образом, one-hot encoding, будучи простым и интуитивно понятным способом представления, страдает от трёх фундаментальных проблем: высокая размерность и разреженность, отсутствие семантической близости, неспособность обрабатывать неизвестные слова. Эти ограничения мотивируют разработку более продвинутых методов векторизации, которые учитывают статистику совместной встречаемости слов и их контекстное распределение.

## 1.6 Роль one-hot encoding в современных архитектурах

Несмотря на перечисленные недостатки, one-hot encoding не исчез из практики. Он широко применяется в нейронных сетях как способ ввода категориальных данных. В частности, в моделях NLP на основе нейронных сетей каждое слово сначала преобразуется в one-hot вектор, который затем умножается на матрицу встраиваний (embedding matrix). Это умножение фактически выбирает строку матрицы, соответствующую индексу слова, и на выходе получается плотный низкоразмерный вектор. Именно эти плотные векторы, а не one-hot векторы, и используются далее в вычислениях. Таким образом, one-hot encoding служит лишь промежуточным звеном, обеспечивающим удобную индексацию, но не является финальным представлением.

Понимание one-hot encoding необходимо для осознания того, почему последующие методы — bag-of-words, TF-IDF, латентный семантический анализ, тематические модели и нейросетевые эмбеддинги — были разработаны и какие проблемы они призваны решить. Каждый из этих подходов добавляет всё больше семантической информации, сохраняя при этом вычислительную эффективность. В следующей части лекции мы перейдём к рассмотрению bag-of-words, который делает первый шаг от представления отдельного слова к представлению целого документа, хотя и ценой потери порядка слов.

# Часть 2. Bag of Words и TF-IDF: представление документов на основе частот

## 2.1 От представления слова к представлению документа

One-hot encoding, рассмотренное в предыдущей части, оперирует отдельными словами и не даёт никакого способа представить целый документ, кроме как в виде последовательности огромных разреженных векторов. Для многих задач — классификации текстов, кластеризации, поиска — необходимо иметь единый вектор фиксированной длины, который описывал бы весь документ. Наиболее естественный и исторически первый способ добиться этого — посчитать, сколько раз каждое слово из общего словаря встречается в данном документе, и записать эти частоты в вектор. Такой подход называется **Bag of Words**, или «мешок слов». Название отражает принципиальное допущение: порядок слов не учитывается, документ рассматривается как неупорядоченный набор слов, как будто все слова высыпали из предложения в мешок и перемешали.

Bag of Words решает проблему переменной длины документа: независимо от того, сколько слов в тексте, его представление всегда имеет размерность, равную размеру общего словаря $|V|$. Это позволяет использовать стандартные алгоритмы машинного обучения, работающие с векторами фиксированной длины. Кроме того, BoW впервые вводит идею, что слова, часто встречающиеся вместе в одних и тех же документах, могут указывать на тематическую близость документов. Например, если в двух документах часто встречаются слова «кошка», «сидит», «окно», то эти документы, вероятно, описывают похожие ситуации, даже если порядок слов различается.

## 2.2 Формальное определение Bag of Words

Пусть задана коллекция документов $D = \{d_1, d_2, \ldots, d_M\}$, где $M$ — количество документов. Из всех документов выделяется словарь $V$ — множество всех уникальных слов, встречающихся в коллекции, и его размер $N = |V|$. Занумеруем слова словаря: $V = \{w_1, w_2, \ldots, w_N\}$. Тогда каждый документ $d_m$ можно представить вектором $\mathbf{x}_m \in \mathbb{R}^N$, компоненты которого равны количеству вхождений соответствующего слова в документ:

$$
(\mathbf{x}_m)_j = \operatorname{tf}(w_j, d_m), \quad j = 1, \ldots, N,
$$

где $\operatorname{tf}(w_j, d_m)$ — частота слова $w_j$ в документе $d_m$, то есть число раз, которое слово $w_j$ встретилось в $d_m$. Такой вектор называется **вектором частот** или **Bag-of-Words вектором**.

Если составить матрицу $X \in \mathbb{R}^{M \times N}$, в которой $m$-я строка является вектором документа $d_m$, то получим **матрицу «документ-термин»**. Элемент $X_{mj}$ равен частоте слова $j$ в документе $m$. Эта матрица содержит всю информацию о частотах слов в коллекции, но не содержит информации о порядке слов внутри документов.

Важно отметить, что BoW-вектор может быть нормализован. Часто применяют L1-нормализацию (деление на сумму частот) или L2-нормализацию (деление на евклидову норму), чтобы уменьшить влияние длины документа. Если документ длинный, то в нём могут встречаться большие абсолютные частоты, что может неоправданно увеличивать его вес при сравнении с короткими документами. Нормализация приводит векторы к сопоставимому масштабу. Однако в базовом определении BoW используется именно сырая частота.

## 2.3 Пример Bag of Words на маленьком корпусе

Рассмотрим тот же учебный корпус, который использовался в части 1:

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».

Словарь этого корпуса состоит из восьми слов: $V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\}$, $N = 8$. Занумеруем слова по порядку: 1 — кошка, 2 — сидит, 3 — на, 4 — окне, 5 — собака, 6 — крыльце, 7 — спит, 8 — диване.

Подсчитаем частоты каждого слова в каждом документе.

Документ 1 содержит слова «кошка», «сидит», «на», «окне» каждое по одному разу. Следовательно, вектор документа 1:

$$
\mathbf{x}_1 = (1, 1, 1, 1, 0, 0, 0, 0)^\top.
$$

Документ 2: «собака», «сидит», «на», «крыльце» по одному разу:

$$
\mathbf{x}_2 = (0, 1, 1, 0, 1, 1, 0, 0)^\top.
$$

Документ 3: «кошка», «спит», «на», «диване» по одному разу:

$$
\mathbf{x}_3 = (1, 0, 1, 0, 0, 0, 1, 1)^\top.
$$

Матрица «документ-термин» $X$ размера $3 \times 8$ выглядит так:

$$
X = \begin{pmatrix}
1 & 1 & 1 & 1 & 0 & 0 & 0 & 0 \\
0 & 1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 1 & 1
\end{pmatrix}.
$$

Видно, что каждая строка соответствует документу, а каждый столбец — слову. Например, первый столбец (слово «кошка») имеет значения 1 в первой строке, 0 во второй, 1 в третьей. Это означает, что «кошка» встречается в документах 1 и 3, но не встречается в документе 2.

Если применить L1-нормализацию (каждый вектор разделить на сумму его компонентов, которая в нашем случае равна 4 для всех документов), то получим векторы:

$$
\mathbf{x}_1^{\text{norm}} = (0.25, 0.25, 0.25, 0.25, 0, 0, 0, 0),
$$
$$
\mathbf{x}_2^{\text{norm}} = (0, 0.25, 0.25, 0, 0.25, 0.25, 0, 0),
$$
$$
\mathbf{x}_3^{\text{norm}} = (0.25, 0, 0.25, 0, 0, 0, 0.25, 0.25).
$$

Такая нормализация делает векторы независимыми от длины документа, но в нашем примере длины совпадают.

## 2.4 Недостатки Bag of Words

Bag of Words страдает от нескольких принципиальных ограничений. Во-первых, полностью игнорируется порядок слов. Выражения «кошка сидит на собаке» и «собака сидит на кошке» дадут одинаковые BoW-векторы, хотя их смысл противоположен. Это не позволяет модели улавливать синтаксис и контекст. Во-вторых, частые служебные слова (например, «на», «и», «в») доминируют в векторах, хотя не несут основной смысловой нагрузки. В нашем примере слово «на» встречается во всех трёх документах и имеет высокую частоту, но оно не помогает различать документы по теме. В-третьих, размерность вектора по-прежнему равна размеру словаря, который в реальных задачах может достигать сотен тысяч, а сами векторы разрежены, что требует эффективных разреженных структур данных.

Кроме того, BoW не отражает семантическую близость слов. Слова «кошка» и «собака» в BoW представлены разными координатами, и их близость не выше, чем близость «кошка» и «диван». Для решения части этих проблем было предложено взвешивание частот слов с учётом их информативности — метод **TF-IDF**.

## 2.5 Идея TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) модифицирует BoW, умножая частоту слова в документе на коэффициент, обратный частоте встречаемости слова во всей коллекции. Основная идея: слова, которые встречаются почти во всех документах (например, предлоги, союзы), несут мало информации для различения документов, поэтому их вес должен быть уменьшен. Напротив, слова, встречающиеся в небольшом числе документов, более специфичны и должны получать больший вес. Таким образом, TF-IDF усиливает значимость редких, но характерных терминов.

TF-IDF не является одной строгой формулой, а представляет семейство взвешивающих схем. В классическом варианте используются две компоненты: **TF** (частота в документе) и **IDF** (обратная документная частота), которые перемножаются.

## 2.6 Формальное определение TF-IDF

Для слова $w_j$ и документа $d_m$ определим:

- **Term Frequency** — частота слова в документе. Это может быть сырая частота $\operatorname{tf}(w_j, d_m)$, но на практике часто используют логарифмированную частоту, чтобы сгладить влияние многократных повторений одного слова. Например, $\operatorname{tf}_{\log}(w_j, d_m) = \log(1 + \operatorname{tf}(w_j, d_m))$.

- **Inverse Document Frequency** — мера редкости слова. Она вычисляется на основе количества документов, в которых слово встречается хотя бы один раз. Обозначим через $df(w_j)$ число документов, содержащих слово $w_j$. Тогда IDF определяется как:

$$
\operatorname{idf}(w_j) = \log \frac{M}{df(w_j)},
$$

где $M$ — общее число документов. Для предотвращения деления на ноль (если слово не встречается ни в одном документе, что невозможно для слов из словаря), иногда используют сглаживание: $\operatorname{idf}(w_j) = \log \frac{M}{1 + df(w_j)}$ или добавляют 1 к числителю и знаменателю. Мы будем использовать базовую формулу без сглаживания, поскольку все слова из словаря встречаются хотя бы один раз.

Тогда вес TF-IDF слова $w_j$ в документе $d_m$ равен произведению:

$$
\operatorname{tf\text{-}idf}(w_j, d_m) = \operatorname{tf}(w_j, d_m) \times \operatorname{idf}(w_j).
$$

Полный вектор документа в пространстве TF-IDF:

$$
\mathbf{x}_m^{\text{tfidf}} = \left( \operatorname{tf\text{-}idf}(w_1, d_m), \ldots, \operatorname{tf\text{-}idf}(w_N, d_m) \right).
$$

Если используется нормализация TF (например, лог-нормализация), то формула принимает вид $\operatorname{tf\text{-}idf}(w_j, d_m) = \operatorname{tf}_{\log}(w_j, d_m) \times \operatorname{idf}(w_j)$.

## 2.7 Пример TF-IDF на том же корпусе

Продолжим работу с нашим корпусом из трёх документов. Имеем $M = 3$. Вычислим IDF для каждого слова.

- Документ 1: «кошка сидит на окне»,
- Документ 2: «собака сидит на крыльце»,
- Документ 3: «кошка спит на диване».


Количество документов, содержащих слово:

- «кошка» встречается в документах 1 и 3, значит $df = 2$.
- «сидит» встречается в документах 1 и 2, $df = 2$.
- «на» встречается во всех трёх документах, $df = 3$.
- «окне» встречается только в документе 1, $df = 1$.
- «собака» только в документе 2, $df = 1$.
- «крыльце» только в документе 2, $df = 1$.
- «спит» только в документе 3, $df = 1$.
- «диване» только в документе 3, $df = 1$.

Теперь вычислим IDF по формуле $\log(M / df)$:

- Для «кошка»: $\log(3/2) \approx 0.405$.
- Для «сидит»: $\log(3/2) \approx 0.405$.
- Для «на»: $\log(3/3) = 0$.
- Для «окне»: $\log(3/1) \approx 1.099$.
- Для «собака»: $\log(3/1) \approx 1.099$.
- Для «крыльце»: $\log(3/1) \approx 1.099$.
- Для «спит»: $\log(3/1) \approx 1.099$.
- Для «диване»: $\log(3/1) \approx 1.099$.

Заметим, что слово «на» получило нулевой IDF, поскольку встречается во всех документах и не помогает их различать. Это решает проблему доминирования частых слов.

Теперь вычислим TF-IDF для каждого документа, используя сырую частоту (все частоты равны 1 для встречающихся слов). Для документа 1:

- кошка: $1 \times 0.405 = 0.405$
- сидит: $1 \times 0.405 = 0.405$
- на: $1 \times 0 = 0$
- окне: $1 \times 1.099 = 1.099$
- остальные: 0.

Вектор документа 1:

$$
\mathbf{x}_1^{\text{tfidf}} = (0.405, 0.405, 0, 1.099, 0, 0, 0, 0).
$$

Документ 2:

- собака: $1 \times 1.099 = 1.099$
- сидит: $0.405$
- на: $0$
- крыльце: $1.099$

$$
\mathbf{x}_2^{\text{tfidf}} = (0, 0.405, 0, 0, 1.099, 1.099, 0, 0).
$$

Документ 3:

- кошка: $0.405$
- спит: $1.099$
- на: $0$
- диване: $1.099$

$$
\mathbf{x}_3^{\text{tfidf}} = (0.405, 0, 0, 0, 0, 0, 1.099, 1.099).
$$

Матрица TF-IDF:

$$
X_{\text{tfidf}} = \begin{pmatrix}
0.405 & 0.405 & 0 & 1.099 & 0 & 0 & 0 & 0 \\
0 & 0.405 & 0 & 0 & 1.099 & 1.099 & 0 & 0 \\
0.405 & 0 & 0 & 0 & 0 & 0 & 1.099 & 1.099
\end{pmatrix}.
$$

Видно, что служебное слово «на» обнулилось, а слова, характерные для одного документа, получили наибольший вес. Это улучшает различение документов.

Если использовать лог-нормализацию TF, $\operatorname{tf}_{\log} = \log(1+1) = \log 2 \approx 0.693$, то веса изменятся: «окне» стало бы $0.693 \times 1.099 \approx 0.762$, но пропорции сохранятся.

## 2.8 Свойства и ограничения TF-IDF

TF-IDF является де-факто стандартом для многих задач информационного поиска и классификации текстов. Он эффективно снижает влияние частых слов и подчёркивает важные термины. Однако он сохраняет основные ограничения BoW: порядок слов игнорируется, семантическая синонимия не учитывается, векторы остаются разреженными и высокоразмерными. Более того, TF-IDF не способен уловить скрытые тематические структуры, которые могут объединять слова, даже если они не встречаются в одних и тех же документах напрямую.

Для преодоления этих ограничений были разработаны методы, основанные на матричной факторизации и вероятностных тематических моделях, о которых пойдёт речь в следующей части. Тем не менее, BoW и TF-IDF остаются важными базовыми инструментами, и понимание их математики необходимо для освоения более сложных подходов.

# Часть 3. Матричные методы на основе совместной встречаемости: HAL и LSA

## 3.1 От частот к контекстам

Представления Bag of Words и TF-IDF, рассмотренные в предыдущей части, основаны на частотах слов в документах и не учитывают связи между словами. Они не позволяют выявить семантическую близость: слова «кошка» и «собака» оказываются такими же далёкими друг от друга, как «кошка» и «диван», хотя интуитивно мы понимаем, что первые два слова семантически связаны. Причина в том, что эти методы представляют каждое слово как отдельную координату, независимую от остальных. Для того чтобы уловить смысловое сходство, необходимо опереться на **дистрибутивную гипотезу**, сформулированную лингвистами ещё в середине XX века: слова, встречающиеся в похожих контекстах, имеют похожие значения. Иными словами, значение слова определяется его окружением.

Эта идея приводит к построению векторных представлений, основанных на **совместной встречаемости** слов в пределах некоторого окна. Если два слова часто появляются рядом с одними и теми же другими словами, их векторы должны быть похожи. Матричные методы, такие как **HAL** (Hyperspace Analogue to Language) и **LSA** (Latent Semantic Analysis), используют этот принцип, строя матрицы совместной встречаемости или матрицы «термин-документ», а затем применяя к ним линейную алгебру для получения плотных векторов слов или документов.

В этой части мы рассмотрим два классических метода: HAL, который строит векторы слов напрямую из матрицы совместной встречаемости без дополнительной факторизации, и LSA, который применяет сингулярное разложение (SVD) к матрице «термин-документ», чтобы выявить латентные семантические факторы.


## 3.2 HAL: Hyperspace Analogue to Language

### 3.2.1 Идея и построение матрицы совместной встречаемости

HAL был предложен Лундом и Бёрджессом в 1996 году как модель семантической памяти, имитирующая то, как человек усваивает значения слов из опыта чтения. Основная идея заключается в том, чтобы для каждого слова в корпусе собрать информацию о его соседях в скользящем окне. Чем чаще два слова встречаются рядом, тем сильнее их семантическая связь. В результате каждое слово получает вектор, компонентами которого являются меры совместной встречаемости с каждым другим словом словаря.

*В этом разделе мы сначала опишем упрощённую симметричную версию HAL, а затем укажем на особенности оригинальной модели, которая различает левый и правый контексты. Это различие важно для точного понимания метода, хотя в учебных примерах для простоты часто используют симметричный вариант.*

Формально, пусть у нас есть корпус, представленный последовательностью слов $w_1, w_2, \ldots, w_T$, где $T$ — общее число словоупотреблений. Зафиксируем размер окна $m$ (например, $m = 1$ или $m = 10$). Для каждой позиции $t$ мы рассматриваем пары $(w_t, w_{t+j})$, где $j \in \{-m, \ldots, -1, 1, \ldots, m\}$. Каждой такой паре приписывается вес, зависящий от расстояния $|j|$: чем ближе слово, тем больше вес. В оригинальной модели HAL используется линейное убывание веса с расстоянием, например $\frac{1}{|j|}$, но для простоты часто берут постоянный вес $1$.

Строится матрица совместной встречаемости $H \in \mathbb{R}^{N \times N}$, где $N = |V|$ — размер словаря. Элемент $H_{ij}$ равен суммарному весу всех вхождений пары $(w_i, w_j)$, где $w_i$ — целевое слово (в позиции $t$), а $w_j$ — контекстное слово (в позиции $t+j$).

*В упрощённой симметричной версии, которую мы будем использовать в примере, мы не различаем левый и правый контекст: $H_{ij}$ учитывает все появления слова $w_j$ в окне вокруг $w_i$ независимо от направления. Тогда матрица $H$ симметрична, так как $H_{ij} = H_{ji}$. Это удобно для первого знакомства с методом, однако оригинальная модель HAL **не является симметричной**.*

*В оригинальной модели HAL информация о левых и правых соседях обрабатывается раздельно. Для каждого слова $w_i$ строится два вектора: один для контекста слева (слова, стоящие до $w_i$ в пределах окна) и один для контекста справа (слова после $w_i$). Эти два вектора затем конкатенируются, образуя итоговый вектор размерности $2N$. Такой подход позволяет сохранить информацию о порядке слов относительно целевого слова, что важно для синтаксических и семантических нюансов.*

*Далее в учебных целях мы будем применять упрощённый симметричный вариант HAL, поскольку он достаточен для иллюстрации базовой идеи и не требует удвоения размерности. При использовании оригинальной несимметричной модели все рассуждения легко обобщаются заменой матрицы $H$ на пару матриц $H^{\text{left}}$ и $H^{\text{right}}$ с последующей конкатенацией их строк.*

Вектором слова $w_i$ в HAL обычно является строка матрицы $H$, то есть вектор $h_i = (H_{i1}, H_{i2}, \ldots, H_{iN})$. Этот вектор показывает, с какими словами и насколько часто встречалось слово $w_i$ в контексте. Часто строки нормализуют (например, L2-норма), чтобы компенсировать разную частоту слов.






## 3.2.2 Пример HAL на нашем корпусе — подробный разбор построения матрицы

### Что такое матрица совместной встречаемости?

Матрица совместной встречаемости $H$ — это квадратная таблица размером $N \times N$, где $N$ — размер словаря. Каждая её строка и каждый столбец соответствуют одному слову из словаря. Элемент $H_{ij}$ показывает, **сколько раз слово $j$ встречалось в окне рядом со словом $i$** (в симметричной версии — сколько раз слова $i$ и $j$ были соседями в пределах заданного окна).

В нашем примере:
- Словарь $V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\}$;
- $N = 8$;
- Нумерация: 1 — кошка, 2 — сидит, 3 — на, 4 — окне, 5 — собака, 6 — крыльце, 7 — спит, 8 — диване.

Размер окна $m=1$, то есть мы рассматриваем только **непосредственных соседей** слева и справа от каждого слова. Вес каждой соседней пары равен 1. Мы **не различаем направление**: если слово A стоит рядом со словом B (слева или справа), мы увеличиваем и $H_{AB}$, и $H_{BA}$ на 1. В результате матрица получается симметричной.

### Исходная последовательность слов

Объединим все документы в одну последовательность (границы документов игнорируем):

$$
\text{кошка}_1 \quad \text{сидит} \quad \text{на} \quad \text{окне} \quad \text{собака} \quad \text{сидит} \quad \text{на} \quad \text{крыльце} \quad \text{кошка}_2 \quad \text{спит} \quad \text{на} \quad \text{диване}
$$

Индексы $_1$ и $_2$ здесь используются только для того, чтобы различать два вхождения слова «кошка»; в матрице оба соответствуют одному индексу 1.

### Построение матрицы по шагам

Проходим по последовательности слева направо. Для каждого слова смотрим на его соседей (если они есть) и увеличиваем соответствующие элементы матрицы.

#### Первое вхождение «кошка» (позиция 1)

- Слева ничего нет.
- Справа — «сидит» (индекс 2).  
  Увеличиваем $H_{1,2}$ на 1 и симметрично $H_{2,1}$ на 1.

После шага: $H_{1,2}=1$, $H_{2,1}=1$, остальные — 0.

#### Слово «сидит» (первое вхождение, позиция 2)

- Слева — «кошка» (уже учтено).
- Справа — «на» (индекс 3).  
  Увеличиваем $H_{2,3}$ на 1 и $H_{3,2}$ на 1.

#### Слово «на» (первое вхождение, позиция 3)

- Слева — «сидит» (учтено).
- Справа — «окне» (индекс 4).  
  Увеличиваем $H_{3,4}$ на 1 и $H_{4,3}$ на 1.

#### Слово «окне» (позиция 4)

- Слева — «на» (учтено).
- Справа — «собака» (индекс 5).  
  Увеличиваем $H_{4,5}$ на 1 и $H_{5,4}$ на 1.

#### Слово «собака» (позиция 5)

- Слева — «окне» (учтено).
- Справа — «сидит» (индекс 2).  
  Увеличиваем $H_{5,2}$ на 1 и $H_{2,5}$ на 1.

Теперь у слова «сидит» (индекс 2) есть два соседства: с «кошка» ($H_{2,1}=1$) и с «собака» ($H_{2,5}=1$).

#### Слово «сидит» (второе вхождение, позиция 6)

- Слева — «собака» (уже учтено на предыдущем шаге, повторно не увеличиваем).
- Справа — «на» (индекс 3).  
  Пара «сидит»–«на» встречается уже второй раз, поэтому увеличиваем $H_{2,3}$ и $H_{3,2}$ ещё на 1.  
  Теперь $H_{2,3}=2$, $H_{3,2}=2$.

#### Слово «на» (второе вхождение, позиция 7)

- Слева — «сидит» (учтено, $H_{3,2}=2$).
- Справа — «крыльце» (индекс 6).  
  Увеличиваем $H_{3,6}$ на 1 и $H_{6,3}$ на 1.

#### Слово «крыльце» (позиция 8)

- Слева — «на» (учтено).
- Справа — «кошка» (индекс 1).  
  Увеличиваем $H_{6,1}$ на 1 и $H_{1,6}$ на 1.  
  Это соседство относится ко второму вхождению «кошки».

#### Второе вхождение «кошка» (позиция 9)

- Слева — «крыльце» (уже учтено, $H_{1,6}=1$).
- Справа — «спит» (индекс 7).  
  Увеличиваем $H_{1,7}$ на 1 и $H_{7,1}$ на 1.

Теперь у «кошки» есть три соседства: с «сидит» (1 раз), с «крыльце» (1 раз), с «спит» (1 раз).

#### Слово «спит» (позиция 10)

- Слева — «кошка» (учтено).
- Справа — «на» (индекс 3).  
  Увеличиваем $H_{7,3}$ на 1 и $H_{3,7}$ на 1.

#### Слово «на» (третье вхождение, позиция 11)

- Слева — «спит» (учтено).
- Справа — «диване» (индекс 8).  
  Увеличиваем $H_{3,8}$ на 1 и $H_{8,3}$ на 1.

#### Слово «диване» (позиция 12)

- Слева — «на» (учтено).
- Справа соседей нет.

### Итоговая матрица $H$

После обработки всех позиций получаем симметричную матрицу $8 \times 8$:

$$
\begin{array}{c|cccccccc}
 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 \\
\hline
1 & 0 & 1 & 0 & 0 & 0 & 1 & 1 & 0 \\
2 & 1 & 0 & 2 & 0 & 1 & 0 & 0 & 0 \\
3 & 0 & 2 & 0 & 1 & 0 & 1 & 1 & 1 \\
4 & 0 & 0 & 1 & 0 & 1 & 0 & 0 & 0 \\
5 & 0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 \\
6 & 1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
7 & 1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
8 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0
\end{array}
$$

Или в более привычном матричном виде:

$$
H = \begin{pmatrix}
0 & 1 & 0 & 0 & 0 & 1 & 1 & 0 \\
1 & 0 & 2 & 0 & 1 & 0 & 0 & 0 \\
0 & 2 & 0 & 1 & 0 & 1 & 1 & 1 \\
0 & 0 & 1 & 0 & 1 & 0 & 0 & 0 \\
0 & 1 & 0 & 1 & 0 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
1 & 0 & 1 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 & 0 & 0
\end{pmatrix}
$$

Матрица симметрична: $H_{ij} = H_{ji}$. Например, $H_{1,2} = H_{2,1} = 1$, $H_{2,3} = H_{3,2} = 2$.

### Что означают строки матрицы?

Каждая строка матрицы — это вектор соответствующего слова.

#### Вектор слова «кошка» (индекс 1)

$$
v_{\text{кошка}} = (0,\ 1,\ 0,\ 0,\ 0,\ 1,\ 1,\ 0)
$$

- $H_{1,2}=1$ — слово «сидит» было соседом «кошки» 1 раз (в самом начале: «кошка сидит»).
- $H_{1,6}=1$ — слово «крыльце» было соседом 1 раз («крыльце кошка»).
- $H_{1,7}=1$ — слово «спит» было соседом 1 раз («кошка спит»).
- Остальные позиции равны нулю.

#### Вектор слова «сидит» (индекс 2)

$$
v_{\text{сидит}} = (1,\ 0,\ 2,\ 0,\ 1,\ 0,\ 0,\ 0)
$$

- $H_{2,1}=1$ — соседство с «кошка».
- $H_{2,3}=2$ — соседство с «на» встречалось дважды («сидит на окне» и «сидит на крыльце»).
- $H_{2,5}=1$ — соседство с «собака» («собака сидит»).

#### Вектор слова «на» (индекс 3)

$$
v_{\text{на}} = (0,\ 2,\ 0,\ 1,\ 0,\ 1,\ 1,\ 1)
$$

Слово «на» часто выступает связкой, поэтому имеет много ненулевых связей: с «сидит» (2 раза), с «окне», «крыльце», «спит», «диване» (по 1 разу).

### Сравнение векторов слов

Теперь, имея векторы, можно вычислять косинусное сходство, чтобы оценить семантическую близость слов, основанную на их контекстах.

Возьмём слова «кошка» и «собака»:

$$
v_{\text{кошка}} = (0,1,0,0,0,1,1,0)
$$
$$
v_{\text{собака}} = (0,1,0,1,0,0,0,0)
$$

Скалярное произведение:

$$
v_{\text{кошка}} \cdot v_{\text{собака}} = 0\cdot0 + 1\cdot1 + 0\cdot0 + 0\cdot1 + 0\cdot0 + 1\cdot0 + 1\cdot0 + 0\cdot0 = 1
$$

Нормы векторов:

$$
\|v_{\text{кошка}}\| = \sqrt{0^2 + 1^2 + 0^2 + 0^2 + 0^2 + 1^2 + 1^2 + 0^2} = \sqrt{3}
$$
$$
\|v_{\text{собака}}\| = \sqrt{0^2 + 1^2 + 0^2 + 1^2 + 0^2 + 0^2 + 0^2 + 0^2} = \sqrt{2}
$$

Косинусное сходство:

$$
\cos(\theta) = \frac{1}{\sqrt{3}\cdot\sqrt{2}} = \frac{1}{\sqrt{6}} \approx 0.408
$$

Ненулевое значение ($0.408$) указывает на то, что слова «кошка» и «собака» имеют некоторое сходство, поскольку оба встречаются рядом со словом «сидит». В one-hot представлении это сходство было бы строго равно нулю.

### Замечания по реализации

- В реальной модели HAL окно обычно больше (например, $m = 4$–$10$), а вес соседства убывает с расстоянием (например, $\frac{1}{|j|}$). Это делает векторы более гладкими и содержательными.
- Часто перед построением матрицы применяют взвешивание, аналогичное TF-IDF, чтобы снизить влияние слишком частых слов (например, предлога «на», у которого много связей).
- В оригинальной модели HAL левый и правый контексты хранятся раздельно и затем конкатенируются; в нашем учебном примере мы использовали симметричную версию для простоты.

Таким образом, матрица совместной встречаемости $H$ является центральным элементом метода HAL: её строки служат векторными представлениями слов, отражающими их контекстную дистрибуцию, что позволяет улавливать семантическую близость.


### 3.2.3 Ограничения HAL

HAL страдает от высокой размерности (равной размеру словаря) и разреженности, поскольку большинство пар слов никогда не встречаются в пределах окна. Кроме того, он чувствителен к частотным словам: если не применять нормализацию, частые слова будут иметь большие значения. Для устранения этих проблем часто применяют взвешивание, аналогичное TF-IDF, или методы понижения размерности, такие как SVD, что приводит нас к LSA.



## 3.3 LSA: Latent Semantic Analysis

### 3.3.1 Мотивация и идея

LSA, предложенный в 1990 году, решает проблему разреженности и высокой размерности, а также частично проблему синонимии, путём применения сингулярного разложения к матрице «термин-документ». Основная идея состоит в том, что между словами и документами существует скрытая (латентная) семантическая структура, которая может быть выявлена с помощью линейной алгебры. LSA проецирует слова и документы в низкоразмерное пространство, где семантически близкие объекты оказываются рядом.

В отличие от HAL, который строит матрицу совместной встречаемости слов, LSA обычно работает с матрицей частот слов в документах (часто взвешенной TF-IDF). Каждая строка этой матрицы соответствует слову, а каждый столбец — документу, либо наоборот. Применяя SVD, мы получаем разложение, которое позволяет аппроксимировать исходную матрицу произведением трёх матриц меньшего ранга. Это разложение выделяет главные направления вариации данных, которые интерпретируются как латентные темы или факторы.

### 3.3.2 Математическая основа SVD

Пусть у нас есть матрица $X \in \mathbb{R}^{m \times n}$, где $m$ — количество строк (например, слов), $n$ — количество столбцов (например, документов). Сингулярное разложение матрицы $X$ имеет вид:

$$
X = U \Sigma V^\top,
$$

где:

- $U \in \mathbb{R}^{m \times m}$ — ортонормированная матрица левых сингулярных векторов, столбцы которой являются собственными векторами матрицы $XX^\top$;
- $\Sigma \in \mathbb{R}^{m \times n}$ — диагональная матрица, на главной диагонали которой стоят сингулярные числа $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$, где $r = \operatorname{rank}(X)$;
- $V \in \mathbb{R}^{n \times n}$ — ортонормированная матрица правых сингулярных векторов, столбцы которой являются собственными векторами матрицы $X^\top X$.

Если взять только $k$ наибольших сингулярных чисел и соответствующие столбцы $U$ и $V$, то получим **усечённое SVD**:

$$
X_k = U_k \Sigma_k V_k^\top,
$$

где $U_k \in \mathbb{R}^{m \times k}$, $\Sigma_k \in \mathbb{R}^{k \times k}$, $V_k \in \mathbb{R}^{n \times k}$. Матрица $X_k$ является наилучшим приближением матрицы $X$ ранга $k$ в смысле как спектральной нормы, так и нормы Фробениуса (теорема Эккарта–Янга). Это означает, что мы можем существенно снизить размерность, сохранив основную структуру данных.

В контексте LSA обычно строят матрицу $X$ размера $N \times M$ (слова × документы) или $M \times N$ (документы × слова). Для определённости будем считать, что строки — слова, столбцы — документы. Тогда:

- Векторы слов в латентном пространстве получаются как строки матрицы $U_k$ (или $U_k \Sigma_k$). Каждая строка соответствует слову и имеет длину $k$.
- Векторы документов получаются как строки матрицы $V_k$ (или $V_k \Sigma_k$). Каждая строка соответствует документу.

Для сравнения слов или документов используют косинусную близость в $k$-мерном пространстве. Часто используют нормализованные векторы $U_k$ и $V_k$, так как сингулярные числа уже учтены в масштабе.

*На практике предпочтительнее использовать \(U_k \Sigma_k\) для векторов слов и \(V_k \Sigma_k\) для векторов документов, поскольку эти матрицы явно включают масштаб сингулярных чисел, отражающий важность каждой латентной размерности. Если же требуется сравнивать слова и документы в одном пространстве (например, запрос как вектор слов против документов), можно использовать \(U_k\) и \(V_k\), но тогда необходимо помнить, что масштаб сингулярных чисел не учтён. В учебных примерах для простоты допустимо работать с \(U_k\) и \(V_k\), если \(k\) невелико и относительные расстояния важнее абсолютных значений.*

### 3.3.3 Пример LSA на нашем корпусе

Используем матрицу TF-IDF из части 2 для нашего корпуса. Напомним, что после TF-IDF мы получили матрицу $X$ размера $8 \times 3$ (слова × документы), где строки соответствуют словам в порядке: кошка, сидит, на, окне, собака, крыльце, спит, диване, а столбцы — документам 1, 2, 3. Матрица имеет вид (округлённо до трёх знаков):

$$
X = \begin{pmatrix}
0.405 & 0 & 0.405 \\
0.405 & 0.405 & 0 \\
0 & 0 & 0 \\
1.099 & 0 & 0 \\
0 & 1.099 & 0 \\
0 & 1.099 & 0 \\
0 & 0 & 1.099 \\
0 & 0 & 1.099
\end{pmatrix}.
$$

Применим SVD к этой матрице. Для ручного вычисления SVD матрицы $8 \times 3$ достаточно трудоёмко, поэтому приведём условные, но правдоподобные результаты, иллюстрирующие идею. Предположим, что мы выбрали $k=2$ латентных измерения. После вычислений получим матрицы $U_2$, $\Sigma_2$, $V_2$ (округлённо):

$$
\Sigma_2 = \begin{pmatrix}
2.0 & 0 \\
0 & 1.5
\end{pmatrix},
$$

$$
U_2 = \begin{pmatrix}
-0.35 & 0.40 \\
-0.30 & -0.50 \\
-0.05 & 0.10 \\
-0.40 & 0.30 \\
-0.35 & -0.40 \\
-0.35 & -0.40 \\
-0.40 & 0.30 \\
-0.40 & 0.30
\end{pmatrix}.
$$

$$
V_2 = \begin{pmatrix}
-0.45 & -0.55 \\
-0.50 & 0.60 \\
-0.55 & -0.35
\end{pmatrix}.
$$

*Для простоты иллюстрации мы будем анализировать непосредственно строки матриц \(U_2\) и \(V_2\), не умножая их на \(\Sigma_2\). В реальных приложениях умножение на \(\Sigma_2\) может изменить относительные масштабы, но для нашего учебного примера это не влияет на качественные выводы о взаимном расположении слов и документов.*

Тогда векторы слов (строки $U_2$) в двумерном пространстве:

- кошка: $(-0.35, 0.40)$
- сидит: $(-0.30, -0.50)$
- на: $(-0.05, 0.10)$
- окне: $(-0.40, 0.30)$
- собака: $(-0.35, -0.40)$
- крыльце: $(-0.35, -0.40)$
- спит: $(-0.40, 0.30)$
- диване: $(-0.40, 0.30)$

Векторы документов (строки $V_2$, можно умножить на $\Sigma_2$, но для сравнения часто используют просто $V_2$):

- Документ 1: $(-0.45, -0.55)$
- Документ 2: $(-0.50, 0.60)$
- Документ 3: $(-0.55, -0.35)$

Интерпретируем полученные результаты. Слово «на» получило вектор близкий к нулю по первой координате и небольшой по второй, что отражает его низкую информативность (IDF=0). Слова «окне», «спит», «диване» имеют одинаковый вектор $(-0.40, 0.30)$, что указывает на то, что они связаны с одной латентной темой (возможно, «действия и места, характерные для кошки»). Слова «собака» и «крыльце» имеют вектор $(-0.35, -0.40)$, группируясь во вторую тему. Слово «кошка» занимает промежуточное положение между этими группами, так как встречается в документах с разными темами. Документы 1 и 3 расположены ближе друг к другу (оба про кошку), чем к документу 2 (про собаку). Это согласуется с интуицией.

Если вычислить косинусную близость между векторами слов «кошка» и «собака», она будет положительной (оба имеют отрицательную первую координату и разные знаки второй), в отличие от нулевой близости в one-hot. Таким образом, LSA улавливает семантическое сходство, основанное на совместной встречаемости в документах.

### 3.3.4 Свойства и ограничения LSA

LSA имеет ряд достоинств: она снижает размерность, устраняет шум, улавливает латентные семантические связи и позволяет сравнивать слова и документы в общем пространстве. Однако у неё есть и ограничения. Во-первых, SVD требует значительных вычислительных ресурсов для больших матриц, хотя существуют эффективные разреженные алгоритмы. *Полное сингулярное разложение матрицы размера \(m \times n\) имеет временную сложность \(O(mn \cdot \min(m,n))\), что для корпусов с сотнями тысяч слов и миллионов документов становится практически неприемлемым. Именно это ограничение стимулировало разработку более масштабируемых вероятностных тематических моделей, таких как pLSA и LDA.* Во-вторых, выбор числа измерений $k$ эвристичен и сильно влияет на результат. В-третьих, LSA не является вероятностной моделью: она не описывает процесс порождения текста и не имеет чёткой статистической интерпретации. Кроме того, LSA плохо работает с полисемией: слово с несколькими значениями усредняется в одном векторе.

Эти ограничения мотивировали разработку вероятностных тематических моделей, таких как pLSA и LDA, которые явно вводят скрытые темы и распределения вероятностей, что будет рассмотрено в следующей части.


# Часть 4.1. Вероятностный латентный семантический анализ (pLSA)

## 4.1.1 От линейной алгебры к вероятностной модели

### Что не так с LSA?

Латентный семантический анализ сделал важный шаг вперёд по сравнению с BoW и TF-IDF: он позволил снизить размерность и выявить скрытые факторы, объясняющие совместную встречаемость слов и документов. Однако LSA остаётся **чисто алгебраическим методом**. Он отвечает на вопрос «как приблизить матрицу $X$ матрицей меньшего ранга», но не отвечает на вопросы:

- **Как порождается текст?**  
  LSA не описывает вероятностный процесс, который мог бы объяснить, почему слова и документы распределены именно так. Он просто раскладывает матрицу на сингулярные векторы, и эти векторы не имеют статистической интерпретации.

- **Что делать с новым документом?**  
  Если после обучения LSA появляется новый документ, мы можем спроецировать его в латентное пространство, но эта проекция не имеет вероятностного смысла. Мы не можем сказать, «какова вероятность этого документа при данной модели».

- **Как выбирать число латентных измерений $k$?**  
  В LSA нет статистического критерия для выбора $k$. Мы выбираем его эвристически, глядя на график сингулярных чисел или на качество downstream-задачи.

- **Как интерпретировать латентные измерения?**  
  Сингулярные векторы — это просто направления в пространстве. Они не являются распределениями вероятностей и не поддаются вероятностной интерпретации.

Эти ограничения мотивировали разработку **вероятностных тематических моделей**, в которых документ рассматривается как смесь скрытых тем, а каждая тема — как распределение вероятностей над словами.

### Первая вероятностная тематическая модель: pLSA

Первой и наиболее простой вероятностной тематической моделью является **вероятностный латентный семантический анализ** (probabilistic Latent Semantic Analysis, pLSA), предложенный Хофманом в 1999 году.

**Основная идея pLSA:** каждое слово в документе порождается некоторой скрытой темой. Мы не наблюдаем темы, но можем восстановить их распределения, если предположим, что документ — это смесь тем, а тема — это распределение слов.

Формально, pLSA вводит скрытую переменную $z$ и описывает порождение каждого слова как двухступенчатый процесс:

1. Сначала выбирается тема $z$ согласно распределению тем документа $P(z \mid d)$.
2. Затем из выбранной темы генерируется слово $w$ согласно распределению слов темы $P(w \mid z)$.

В отличие от LSA, pLSA имеет явную вероятностную интерпретацию и обучается методом максимального правдоподобия. Это делает его мостом между алгебраическими методами (LSA) и полностью байесовскими моделями (LDA).

---

## 4.1.2 Порождающая модель и параметры

### Что такое порождающая модель?

**Порождающая модель** (generative model) — это вероятностная модель, которая описывает, как данные могли быть порождены. В контексте NLP порождающая модель текста описывает процесс, который, если его запустить, создаст коллекцию документов, похожую на наблюдаемую.

Важно понимать: мы не утверждаем, что реальный автор текста действительно бросает монетку, чтобы выбрать тему, а затем другую монетку, чтобы выбрать слово. Порождающая модель — это **математическая фикция**, удобная для формулировки задачи. Если такая модель хорошо описывает данные, значит, она уловила важные закономерности.

### Формальное описание

Пусть коллекция состоит из $M$ документов, а словарь содержит $N$ уникальных слов. Введём скрытую переменную

$$
z \in \{1, 2, \dots, K\},
$$

где $K$ — **заданное** число тем.

**Тонкий момент:** $K$ — это гиперпараметр, который выбирает исследователь, а не модель. Мы не «находим» число тем из данных (по крайней мере, в стандартной pLSA). Мы говорим алгоритму: «Ищи $K$ тем», и он ищет ровно $K$ тем. Если $K$ слишком мало, темы будут смешанными. Если слишком много — темы будут дублироваться или становиться бессмысленными. Выбор $K$ — отдельная задача, о которой мы поговорим ниже.

### Распределение тем в документе

Каждый документ $d$ характеризуется **распределением вероятностей тем**:

$$
\theta_{dz} = P(z \mid d), \quad \sum_{z=1}^{K} \theta_{dz} = 1, \quad \theta_{dz} \ge 0.
$$

Это вектор длины $K$, который показывает, какая доля слов в документе $d$ порождена каждой темой. Например, если $\theta_{d,1} = 0.7$ и $\theta_{d,2} = 0.3$, это значит, что 70% слов документа $d$ относятся к теме 1 и 30% — к теме 2.

**Интуиция:** представьте, что документ — это мешок слов, и каждое слово мы вытаскиваем так: сначала бросаем «тематическую монетку» с вероятностями $\theta_d$, а затем, в зависимости от выпавшей темы, бросаем «словесную монетку» с вероятностями $\phi_z$.

### Распределение слов в теме

Каждая тема $z$ характеризуется **распределением вероятностей слов**:

$$
\phi_{zw} = P(w \mid z), \quad \sum_{w \in V} \phi_{zw} = 1, \quad \phi_{zw} \ge 0.
$$

Это вектор длины $N$, который показывает, какие слова характерны для данной темы. Например, если тема «животные», то $\phi_{z,\text{кошка}}$ и $\phi_{z,\text{собака}}$ будут высокими, а $\phi_{z,\text{диван}}$ — низким.

### Двухступенчатый процесс порождения

Предполагается, что порождение каждого слова в документе происходит следующим образом:

1. **Выбор темы.** Для каждой позиции слова в документе $d$ сначала выбирается тема $z$ согласно распределению $P(z \mid d)$.

2. **Выбор слова.** Из выбранной темы $z$ генерируется слово $w$ согласно распределению $P(w \mid z)$.

Формально, вероятность появления слова $w$ в документе $d$ записывается как

$$
P(w, d) = P(d) \sum_{z=1}^{K} P(w \mid z) P(z \mid d).
$$

**Разберём эту формулу по частям:**

- $P(d)$ — априорная вероятность документа. Обычно её не параметризуют и считают заданной эмпирической частотой (например, $P(d) = 1/M$, если все документы равновероятны). В дальнейшем мы будем опускать этот множитель, так как он не влияет на оценку параметров $\theta$ и $\phi$.

- $P(z \mid d)$ — вероятность выбрать тему $z$ в документе $d$. Это параметр $\theta_{dz}$.

- $P(w \mid z)$ — вероятность выбрать слово $w$ в теме $z$. Это параметр $\phi_{zw}$.

- Сумма по $z$ — маргинализация по скрытой переменной. Мы не наблюдаем тему $z$, поэтому суммируем по всем возможным темам.

### Вывод формулы через маргинализацию

Формально, совместная вероятность слова и документа при известной теме:

$$
P(w, z, d) = P(d) P(z \mid d) P(w \mid z, d).
$$

Если предположить **условную независимость** слова от документа при фиксированной теме:

$$
P(w \mid z, d) = P(w \mid z),
$$

то получаем:

$$
P(w, z, d) = P(d) P(z \mid d) P(w \mid z).
$$

Маргинализуя по $z$ (то есть суммируя по всем возможным темам), получаем:

$$
P(w, d) = \sum_{z=1}^{K} P(w, z, d) = P(d) \sum_{z=1}^{K} P(w \mid z) P(z \mid d).
$$

**Тонкий момент:** условие $P(w \mid z, d) = P(w \mid z)$ — это ключевое предположение модели. Оно означает, что если мы знаем тему, то документ не даёт дополнительной информации о слове. Именно это предположение позволяет факторизовать модель и делает pLSA вычислительно эффективной.

### Параметры модели

Параметрами модели являются два набора условных распределений:

- $\theta_{dz} = P(z \mid d)$ — распределение тем в документе $d$;
- $\phi_{zw} = P(w \mid z)$ — распределение слов в теме $z$.

Общее число параметров:

$$
\underbrace{M \times K}_{\theta} + \underbrace{K \times N}_{\phi}.
$$

**Тонкий момент:** при $K \ll \min(M, N)$ это значительно меньше, чем размерность матрицы «термин-документ» ($M \times N$). Однако число параметров $\theta$ растёт **линейно с числом документов** $M$. Это означает, что для каждого нового документа появляется $K$ новых параметров. При большом $M$ это приводит к переобучению. Именно этот недостаток pLSA мотивировал переход к LDA, где $\theta_d$ — не параметры, а случайные величины, порождённые из общего априорного распределения.

### Связь с матричной факторизацией

pLSA можно рассматривать как **вероятностную матричную факторизацию**. Действительно, матрица $P(w \mid d)$ размера $N \times M$ аппроксимируется произведением двух матриц:

$$
P(w \mid d) = \sum_{z=1}^{K} P(w \mid z) P(z \mid d) = \Phi \, \Theta^\top,
$$

где $\Phi$ — матрица $N \times K$ (темы × слова), а $\Theta$ — матрица $M \times K$ (документы × темы). Это похоже на LSA, но с двумя важными отличиями:

1. Все элементы неотрицательны (вероятности).
2. Факторизация имеет вероятностную интерпретацию.

Это наблюдение связывает pLSA с неотрицательной матричной факторизацией (NMF) и показывает, что тематические модели — это частный случай разложения матриц.

---

## 4.1.3 Функция правдоподобия

### Почему правдоподобие?

Мы хотим найти такие параметры $\theta$ и $\phi$, при которых наблюдаемые данные (коллекция документов) наиболее вероятны. Это принцип **максимального правдоподобия** (Maximum Likelihood Estimation, MLE). Он не единственный возможный (есть ещё байесовский подход, который мы увидим в LDA), но самый простой и естественный для pLSA.

### Полное правдоподобие

Пусть $n(d, w)$ — частота слова $w$ в документе $d$. В нашем учебном примере $n(d, w)$ равна либо 0, либо 1, так как каждое слово встречается не более одного раза. В общем случае это неотрицательное целое число.

Если предположить, что все словоупотребления независимы (при фиксированных параметрах), то полное правдоподобие коллекции:

$$
\mathcal{L}(\theta, \phi) = \prod_{d=1}^{M} \prod_{w \in V} P(w, d)^{n(d, w)}.
$$

**Тонкий момент:** здесь мы используем $P(w, d)$, а не $P(w \mid d)$. Это связано с тем, что в порождающей модели документ и слово порождаются совместно. Однако $P(d)$ фиксирована, поэтому максимизация $\mathcal{L}$ по $\theta$ и $\phi$ эквивалентна максимизации правдоподобия $P(w \mid d)$.

### Логарифм правдоподобия

Произведение вероятностей неудобно оптимизировать, поэтому переходим к логарифму. Поскольку логарифм — монотонно возрастающая функция, максимум $\mathcal{L}$ и максимум $\log \mathcal{L}$ достигаются при одних и тех же параметрах.

$$
\ell(\theta, \phi) = \log \mathcal{L} = \sum_{d=1}^{M} \sum_{w \in V} n(d, w) \log P(w, d).
$$

Подставляя выражение для $P(w, d)$ и опуская $P(d)$ (так как она не зависит от параметров), получаем:

$$
\ell(\theta, \phi) = \sum_{d=1}^{M} \sum_{w \in V} n(d, w) \log \left( \sum_{z=1}^{K} P(w \mid z) P(z \mid d) \right).
$$

**Тонкий момент:** сумма по $z$ стоит **внутри логарифма**. Это ключевая сложность pLSA. Если бы темы были наблюдаемыми, логарифм можно было бы «протащить» внутрь суммы, и задача свелась бы к простым формулам. Но темы скрыты, поэтому мы имеем дело с логарифмом суммы — а это невыпуклая функция, которую трудно оптимизировать напрямую. Именно поэтому используется EM-алгоритм.

### Ограничения на параметры

Параметры должны удовлетворять вероятностным ограничениям:

$$
\sum_{w \in V} \phi_{zw} = 1 \quad \text{для всех } z,
$$
$$
\sum_{z=1}^{K} \theta_{dz} = 1 \quad \text{для всех } d,
$$
$$
\phi_{zw} \ge 0, \quad \theta_{dz} \ge 0.
$$

Эти ограничения называются **симплексными**. Они означают, что $\phi_z$ и $\theta_d$ лежат на вероятностном симплексе размерности $N-1$ и $K-1$ соответственно.

### Проблема идентифицируемости

pLSA имеет **проблему идентифицируемости**: если переставить темы местами (например, поменять $z=1$ и $z=2$), значение правдоподобия не изменится. Это означает, что решение не единственно: мы можем получить эквивалентные модели с разными метками тем. На практике это не страшно — мы просто интерпретируем темы после обучения, — но это важно помнить при сравнении моделей.

Кроме того, pLSA может сходиться к разным локальным максимумам при разной инициализации. EM гарантирует неубывание правдоподобия, но не гарантирует глобальный максимум.

---

## 4.1.4 EM-алгоритм для pLSA: полный вывод

### Зачем нужен EM?

Мы хотим максимизировать $\ell(\theta, \phi)$, но в формуле стоит логарифм суммы. Если бы мы знали темы $z$ для каждого слова, задача была бы простой: мы могли бы посчитать частоты и получить оценки максимального правдоподобия. Но темы скрыты.

**EM-алгоритм** (Expectation-Maximization) — это итеративный метод для максимизации правдоподобия в моделях со скрытыми переменными. Идея:

1. **E-шаг:** при текущих параметрах вычислить апостериорное распределение скрытых переменных $P(z \mid d, w)$.
2. **M-шаг:** используя это распределение, обновить параметры так, чтобы максимизировать **ожидаемое** полное правдоподобие.

EM гарантирует, что наблюдаемое правдоподобие $\ell$ не убывает на каждой итерации. Это следует из неравенства Йенсена и того факта, что EM максимизирует нижнюю оценку $\ell$.

### Вывод EM через неравенство Йенсена

Рассмотрим логарифм правдоподобия:

$$
\ell(\theta, \phi) = \sum_{d, w} n(d, w) \log \left( \sum_{z} P(w \mid z) P(z \mid d) \right).
$$

Введём произвольное распределение $q(z \mid d, w)$ по темам для каждой пары $(d, w)$, такое что $\sum_z q(z \mid d, w) = 1$ и $q(z \mid d, w) \ge 0$. Тогда:

$$
\log \left( \sum_{z} P(w \mid z) P(z \mid d) \right) = \log \left( \sum_{z} q(z \mid d, w) \frac{P(w \mid z) P(z \mid d)}{q(z \mid d, w)} \right).
$$

По неравенству Йенсена для вогнутой функции $\log$:

$$
\log \left( \sum_{z} q(z \mid d, w) \frac{P(w \mid z) P(z \mid d)}{q(z \mid d, w)} \right) \ge \sum_{z} q(z \mid d, w) \log \left( \frac{P(w \mid z) P(z \mid d)}{q(z \mid d, w)} \right).
$$

Таким образом, мы получаем **нижнюю оценку** для $\ell$:

$$
\ell(\theta, \phi) \ge \sum_{d, w} n(d, w) \sum_{z} q(z \mid d, w) \log \left( \frac{P(w \mid z) P(z \mid d)}{q(z \mid d, w)} \right) = \mathcal{F}(q, \theta, \phi).
$$

**Ключевая идея EM:** чередовать максимизацию $\mathcal{F}$ по $q$ (E-шаг) и по $\theta, \phi$ (M-шаг).

### E-шаг: оптимальное $q$

На E-шаге мы максимизируем $\mathcal{F}$ по $q$ при фиксированных $\theta, \phi$. Можно показать, что максимум достигается при:

$$
q(z \mid d, w) = P(z \mid d, w) = \frac{P(w \mid z) P(z \mid d)}{\sum_{z'} P(w \mid z') P(z' \mid d)}.
$$

**Вывод:** по формуле Байеса,

$$
P(z \mid d, w) = \frac{P(w, z \mid d)}{P(w \mid d)} = \frac{P(w \mid z) P(z \mid d)}{\sum_{z'} P(w \mid z') P(z' \mid d)}.
$$

Здесь мы использовали:
- условную независимость $P(w \mid z, d) = P(w \mid z)$;
- маргинализацию $P(w \mid d) = \sum_{z'} P(w \mid z') P(z' \mid d)$.

**Тонкий момент:** $P(z \mid d, w)$ называется **ответственностью** (responsibility) темы $z$ за слово $w$ в документе $d$. Она показывает, насколько тема $z$ «виновата» в появлении этого слова. Сумма ответственностей по всем темам равна 1:

$$
\sum_{z=1}^{K} P(z \mid d, w) = 1.
$$

Это вероятность, а не жёсткое назначение. В отличие от кластеризации, где каждое слово принадлежит ровно одному кластеру, в pLSA каждое слово «распределено» по темам с разными весами.

### M-шаг: максимизация $\mathcal{F}$ по параметрам

На M-шаге мы максимизируем $\mathcal{F}$ по $\theta, \phi$ при фиксированном $q(z \mid d, w) = P(z \mid d, w)$. Подставляя это в $\mathcal{F}$ и опуская слагаемые, не зависящие от параметров, получаем:

$$
Q(\theta, \phi) = \sum_{d, w} n(d, w) \sum_{z} P(z \mid d, w) \log \left[ P(w \mid z) P(z \mid d) \right].
$$

Это и есть **ожидаемое полное логарифмическое правдоподобие**.

#### Обновление $\phi_{zw} = P(w \mid z)$

Зафиксируем $z$ и будем максимизировать $Q$ по $\phi_{zw}$ при ограничении $\sum_w \phi_{zw} = 1$. Используем метод множителей Лагранжа. Лагранжиан:

$$
\mathcal{L}_\phi = \sum_{d} \sum_{w} n(d, w) P(z \mid d, w) \log \phi_{zw} + \lambda_z \left( 1 - \sum_w \phi_{zw} \right).
$$

Берём производную по $\phi_{zw}$ и приравниваем к нулю:

$$
\frac{\partial \mathcal{L}_\phi}{\partial \phi_{zw}} = \frac{\sum_d n(d, w) P(z \mid d, w)}{\phi_{zw}} - \lambda_z = 0.
$$

Отсюда:

$$
\phi_{zw} = \frac{\sum_d n(d, w) P(z \mid d, w)}{\lambda_z}.
$$

Используем ограничение $\sum_w \phi_{zw} = 1$:

$$
\sum_w \phi_{zw} = \frac{\sum_w \sum_d n(d, w) P(z \mid d, w)}{\lambda_z} = 1 \implies \lambda_z = \sum_d \sum_w n(d, w) P(z \mid d, w).
$$

Таким образом:

$$
P(w \mid z) = \frac{\sum_{d=1}^{M} n(d, w) P(z \mid d, w)}{\sum_{d=1}^{M} \sum_{w' \in V} n(d, w') P(z \mid d, w')}.
$$

**Интерпретация:** числитель — это ожидаемое число раз, когда слово $w$ было порождено темой $z$. Знаменатель — ожидаемое общее число слов, порождённых темой $z$. Таким образом, $P(w \mid z)$ — это нормированная частота слова $w$ в теме $z$.

#### Обновление $\theta_{dz} = P(z \mid d)$

Аналогично, зафиксируем $d$ и максимизируем $Q$ по $\theta_{dz}$ при ограничении $\sum_z \theta_{dz} = 1$. Лагранжиан:

$$
\mathcal{L}_\theta = \sum_{w} \sum_{z} n(d, w) P(z \mid d, w) \log \theta_{dz} + \mu_d \left( 1 - \sum_z \theta_{dz} \right).
$$

Производная по $\theta_{dz}$:

$$
\frac{\partial \mathcal{L}_\theta}{\partial \theta_{dz}} = \frac{\sum_w n(d, w) P(z \mid d, w)}{\theta_{dz}} - \mu_d = 0.
$$

Отсюда:

$$
\theta_{dz} = \frac{\sum_w n(d, w) P(z \mid d, w)}{\mu_d}.
$$

Используем $\sum_z \theta_{dz} = 1$:

$$
\mu_d = \sum_z \sum_w n(d, w) P(z \mid d, w) = \sum_w n(d, w) \sum_z P(z \mid d, w) = \sum_w n(d, w) = N_d,
$$

где $N_d$ — длина документа $d$ (общее число словоупотреблений). Таким образом:

$$
P(z \mid d) = \frac{\sum_{w \in V} n(d, w) P(z \mid d, w)}{\sum_{w \in V} n(d, w)}.
$$

**Интерпретация:** числитель — ожидаемое число слов в документе $d$, порождённых темой $z$. Знаменатель — общее число слов в документе. Таким образом, $P(z \mid d)$ — это доля слов документа $d$, относящихся к теме $z$.

### Доказательство неубывания правдоподобия

EM гарантирует, что $\ell(\theta^{(t+1)}, \phi^{(t+1)}) \ge \ell(\theta^{(t)}, \phi^{(t)})$ на каждой итерации. Докажем это.

Пусть $q^{(t)}(z \mid d, w) = P(z \mid d, w; \theta^{(t)}, \phi^{(t)})$. Тогда:

$$
\ell(\theta^{(t)}, \phi^{(t)}) = \mathcal{F}(q^{(t)}, \theta^{(t)}, \phi^{(t)}).
$$

Это равенство выполняется, потому что при $q = P(z \mid d, w)$ неравенство Йенсена обращается в равенство (это свойство апостериорного распределения).

На M-шаге мы выбираем $\theta^{(t+1)}, \phi^{(t+1)}$ так, чтобы:

$$
\mathcal{F}(q^{(t)}, \theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t)}, \theta^{(t)}, \phi^{(t)}).
$$

На E-шаге мы выбираем $q^{(t+1)}$ так, чтобы:

$$
\mathcal{F}(q^{(t+1)}, \theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t)}, \theta^{(t+1)}, \phi^{(t+1)}).
$$

Кроме того, для любого $q$:

$$
\ell(\theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t+1)}, \theta^{(t+1)}, \phi^{(t+1)}).
$$

Объединяя все неравенства:

$$
\ell(\theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t+1)}, \theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t)}, \theta^{(t+1)}, \phi^{(t+1)}) \ge \mathcal{F}(q^{(t)}, \theta^{(t)}, \phi^{(t)}) = \ell(\theta^{(t)}, \phi^{(t)}).
$$

Таким образом, правдоподобие не убывает. Это и есть доказательство сходимости EM.

### Сводка EM-алгоритма

1. **Инициализация:** задать начальные $\theta_{dz}$ и $\phi_{zw}$ (случайно или эвристически), нормировать.
2. **E-шаг:** для всех $d, w$ с $n(d, w) > 0$ вычислить
   $$
   P(z \mid d, w) = \frac{P(w \mid z) P(z \mid d)}{\sum_{z'} P(w \mid z') P(z' \mid d)}.
   $$
3. **M-шаг:** обновить
   $$
   P(w \mid z) = \frac{\sum_d n(d, w) P(z \mid d, w)}{\sum_d \sum_{w'} n(d, w') P(z \mid d, w')},
   $$
   $$
   P(z \mid d) = \frac{\sum_w n(d, w) P(z \mid d, w)}{\sum_w n(d, w)}.
   $$
4. **Проверка сходимости:** вычислить логарифм правдоподобия
   $$
   \ell = \sum_d \sum_w n(d, w) \log \left( \sum_z P(w \mid z) P(z \mid d) \right).
   $$
   Если $\ell$ перестал расти (или рост меньше порога), остановиться. Иначе вернуться к шагу 2.

### Тонкие моменты EM-алгоритма

1. **Сходимость к локальному максимуму.**  
   EM гарантирует, что $\ell$ не убывает, но сходится к **локальному** максимуму. Разная инициализация может привести к разным результатам. На практике запускают EM несколько раз с разными начальными значениями и выбирают решение с наибольшим правдоподобием.

2. **Медленная сходимость.**  
   EM может сходиться медленно, особенно вблизи максимума. Иногда используют ускоренные варианты (например, EM с momentum).

3. **Проблема вырождения.**  
   Если тема $z$ «схлопывается» на одно слово $w$, то $P(w \mid z) = 1$, а $P(w' \mid z) = 0$ для всех $w' \neq w$. Это может привести к бесконечному правдоподобию, если $n(d, w) > 0$. На практике этого не происходит из-за нормировки и конечности данных, но в теории такая возможность есть. В LDA эта проблема решается байесовским сглаживанием.

4. **Связь с нижней оценкой.**  
   EM можно интерпретировать как максимизацию нижней оценки $\ell$. На E-шаге мы вычисляем апостериорное распределение $P(z \mid d, w)$, которое даёт наиболее точную нижнюю оценку при текущих параметрах. На M-шаге мы максимизируем эту оценку. Это объясняет, почему EM не может уменьшить $\ell$.

5. **Инициализация.**  
   Начальные значения $\theta$ и $\phi$ могут сильно влиять на результат. В реальных реализациях часто используют случайную инициализацию с нормировкой или инициализацию на основе частот слов. В учебном примере мы зададим начальные значения вручную, чтобы проиллюстрировать первые шаги.

### Что дальше?

Мы разобрали математические основы pLSA: порождающую модель, функцию правдоподобия и EM-алгоритм с полным выводом. Теперь мы готовы применить эти формулы к конкретному корпусу из трёх документов. В следующем разделе мы шаг за шагом проведём E- и M-шаги, вычислим логарифм правдоподобия до и после итерации и покажем, как pLSA автоматически выявляет темы в данных.



## 4.1.5 Численный пример pLSA на учебном корпусе

### 4.1.5.1 Постановка задачи

Рассмотрим тот же корпус из трёх документов, который использовался в предыдущих частях:

- $d_1$: «кошка сидит на окне»;
- $d_2$: «собака сидит на крыльце»;
- $d_3$: «кошка спит на диване».

Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N=8$, число документов $M=3$. Зададим число тем $K=2$.

**Напоминание:** $K=2$ — это наш выбор. Модель не определяет число тем автоматически. Мы могли бы взять $K=3$ или $K=4$, и тогда результат был бы другим. Выбор $K$ — отдельная задача, которую решают по метрикам (перплексия, тематическая когерентность) и по интерпретируемости.

Исходные данные удобно записать в виде матрицы «документ–слово» $n(d,w)$, где $n(d,w)$ — сколько раз слово $w$ встретилось в документе $d$. В нашем случае все частоты равны 0 или 1, потому что каждое слово встречается в каждом документе не более одного раза.

| Документ | кошка | сидит | на | окне | собака | крыльце | спит | диване |
|----------|-------|-------|----|------|--------|---------|------|--------|
| $d_1$    | 1     | 1     | 1  | 1    | 0      | 0       | 0    | 0      |
| $d_2$    | 0     | 1     | 1  | 0    | 1      | 1       | 0    | 0      |
| $d_3$    | 1     | 0     | 1  | 0    | 0      | 0       | 1    | 1      |

Всего в коллекции $12$ словоупотреблений.

**Что мы хотим найти?** Мы хотим найти такие распределения $P(w \mid z)$ и $P(z \mid d)$, при которых наблюдаемые данные наиболее вероятны. Иными словами, мы ищем две темы, которые наилучшим образом объясняют, почему слова распределены по документам именно так.

### 4.1.5.2 Инициализация параметров

EM-алгоритм начинает с некоторого начального приближения. В реальных реализациях начальные значения часто задают случайно и нормируют. В учебном примере мы зададим их вручную, чтобы показать первые шаги EM. Эти числа **не являются «правильными»** — это просто стартовая точка. EM всё равно сойдётся, возможно, к другому локальному максимуму.

**Начальное распределение слов в теме 1 ($z=1$):**

| слово | P(w \| z=1) |
|-------|-----------------|
| кошка | 0.25 |
| сидит | 0.20 |
| на    | 0.08 |
| окне  | 0.03 |
| собака | 0.25 |
| крыльце | 0.02 |
| спит  | 0.15 |
| диване | 0.02 |

Проверка суммы:

$$
0.25+0.20+0.08+0.03+0.25+0.02+0.15+0.02 = 1.00.
$$

**Начальное распределение слов в теме 2 ($z=2$):**

| слово | P(w \| z=2) |
|-------|-----------------|
| кошка | 0.03 |
| сидит | 0.05 |
| на    | 0.30 |
| окне  | 0.25 |
| собака | 0.01 |
| крыльце | 0.20 |
| спит  | 0.01 |
| диване | 0.15 |

Проверка суммы:

$$
0.03+0.05+0.30+0.25+0.01+0.20+0.01+0.15 = 1.00.
$$

**Начальное распределение тем по документам:**

- $P(z=1 \mid d_1) = 0.6$, $P(z=2 \mid d_1) = 0.4$;
- $P(z=1 \mid d_2) = 0.3$, $P(z=2 \mid d_2) = 0.7$;
- $P(z=1 \mid d_3) = 0.55$, $P(z=2 \mid d_3) = 0.45$.

**Интуиция:** эти начальные значения говорят, что, например, в документе $d_1$ 60% слов мы относим к теме 1 и 40% — к теме 2. Тема 1 больше связана с животными («кошка» 0.25, «собака» 0.25, «спит» 0.15), а тема 2 — с местами и предлогом «на» («на» 0.30, «окне» 0.25, «крыльце» 0.20).

### 4.1.5.3 E-шаг: вычисление ответственностей

На E-шаге для каждой пары $(d,w)$, где слово $w$ встречается в документе $d$, мы вычисляем **апостериорную вероятность** того, что это слово порождено темой $z$. Эта вероятность называется **ответственностью** темы за слово:

$$
P(z \mid d, w) = \frac{P(w \mid z) P(z \mid d)}{P(w \mid z=1) P(z=1 \mid d) + P(w \mid z=2) P(z=2 \mid d)}.
$$

**Вывод формулы:** по формуле Байеса,

$$
P(z \mid d, w) = \frac{P(w, z \mid d)}{P(w \mid d)} = \frac{P(w \mid z) P(z \mid d)}{\sum_{z'} P(w \mid z') P(z' \mid d)}.
$$

Здесь мы использовали:
- условную независимость $P(w \mid z, d) = P(w \mid z)$;
- маргинализацию $P(w \mid d) = \sum_{z'} P(w \mid z') P(z' \mid d)$.

Проведём вычисления для всех 12 пар.

**Для $d_1, w=\text{кошка}$:**

$$
P(z=1) = \frac{0.25 \cdot 0.6}{0.25 \cdot 0.6 + 0.03 \cdot 0.4} = \frac{0.15}{0.15 + 0.012} = \frac{0.15}{0.162} \approx 0.9259,
$$

$$
P(z=2) = 1 - 0.9259 = 0.0741.
$$

**Для $d_1, w=\text{сидит}$:**

$$
P(z=1) = \frac{0.20 \cdot 0.6}{0.20 \cdot 0.6 + 0.05 \cdot 0.4} = \frac{0.12}{0.12 + 0.02} = \frac{0.12}{0.14} \approx 0.8571,
$$

$$
P(z=2) \approx 0.1429.
$$

**Для $d_1, w=\text{на}$:**

$$
P(z=1) = \frac{0.08 \cdot 0.6}{0.08 \cdot 0.6 + 0.30 \cdot 0.4} = \frac{0.048}{0.048 + 0.12} = \frac{0.048}{0.168} \approx 0.2857,
$$

$$
P(z=2) \approx 0.7143.
$$

**Для $d_1, w=\text{окне}$:**

$$
P(z=1) = \frac{0.03 \cdot 0.6}{0.03 \cdot 0.6 + 0.25 \cdot 0.4} = \frac{0.018}{0.018 + 0.10} = \frac{0.018}{0.118} \approx 0.1525,
$$

$$
P(z=2) \approx 0.8475.
$$

**Для $d_2, w=\text{собака}$:**

$$
P(z=1) = \frac{0.25 \cdot 0.3}{0.25 \cdot 0.3 + 0.01 \cdot 0.7} = \frac{0.075}{0.075 + 0.007} = \frac{0.075}{0.082} \approx 0.9146,
$$

$$
P(z=2) \approx 0.0854.
$$

**Для $d_2, w=\text{сидит}$:**

$$
P(z=1) = \frac{0.20 \cdot 0.3}{0.20 \cdot 0.3 + 0.05 \cdot 0.7} = \frac{0.06}{0.06 + 0.035} = \frac{0.06}{0.095} \approx 0.6316,
$$

$$
P(z=2) \approx 0.3684.
$$

**Для $d_2, w=\text{на}$:**

$$
P(z=1) = \frac{0.08 \cdot 0.3}{0.08 \cdot 0.3 + 0.30 \cdot 0.7} = \frac{0.024}{0.024 + 0.21} = \frac{0.024}{0.234} \approx 0.1026,
$$

$$
P(z=2) \approx 0.8974.
$$

**Для $d_2, w=\text{крыльце}$:**

$$
P(z=1) = \frac{0.02 \cdot 0.3}{0.02 \cdot 0.3 + 0.20 \cdot 0.7} = \frac{0.006}{0.006 + 0.14} = \frac{0.006}{0.146} \approx 0.0411,
$$

$$
P(z=2) \approx 0.9589.
$$

**Для $d_3, w=\text{кошка}$:**

$$
P(z=1) = \frac{0.25 \cdot 0.55}{0.25 \cdot 0.55 + 0.03 \cdot 0.45} = \frac{0.1375}{0.1375 + 0.0135} = \frac{0.1375}{0.151} \approx 0.9106,
$$

$$
P(z=2) \approx 0.0894.
$$

**Для $d_3, w=\text{спит}$:**

$$
P(z=1) = \frac{0.15 \cdot 0.55}{0.15 \cdot 0.55 + 0.01 \cdot 0.45} = \frac{0.0825}{0.0825 + 0.0045} = \frac{0.0825}{0.087} \approx 0.9483,
$$

$$
P(z=2) \approx 0.0517.
$$

**Для $d_3, w=\text{на}$:**

$$
P(z=1) = \frac{0.08 \cdot 0.55}{0.08 \cdot 0.55 + 0.30 \cdot 0.45} = \frac{0.044}{0.044 + 0.135} = \frac{0.044}{0.179} \approx 0.2458,
$$

$$
P(z=2) \approx 0.7542.
$$

**Для $d_3, w=\text{диване}$:**

$$
P(z=1) = \frac{0.02 \cdot 0.55}{0.02 \cdot 0.55 + 0.15 \cdot 0.45} = \frac{0.011}{0.011 + 0.0675} = \frac{0.011}{0.0785} \approx 0.1401,
$$

$$
P(z=2) \approx 0.8599.
$$

Сведём результаты в таблицу (округление до 4 знаков).

**Таблица E-шага после инициализации**

| Документ | Слово | $P(z=1 \mid d,w)$ | $P(z=2 \mid d,w)$ |
|----------|-------|-------------------|-------------------|
| $d_1$ | кошка | 0.9259 | 0.0741 |
| $d_1$ | сидит | 0.8571 | 0.1429 |
| $d_1$ | на    | 0.2857 | 0.7143 |
| $d_1$ | окне  | 0.1525 | 0.8475 |
| $d_2$ | собака | 0.9146 | 0.0854 |
| $d_2$ | сидит | 0.6316 | 0.3684 |
| $d_2$ | на    | 0.1026 | 0.8974 |
| $d_2$ | крыльце | 0.0411 | 0.9589 |
| $d_3$ | кошка | 0.9106 | 0.0894 |
| $d_3$ | спит  | 0.9483 | 0.0517 |
| $d_3$ | на    | 0.2458 | 0.7542 |
| $d_3$ | диване | 0.1401 | 0.8599 |

**Проверка:** для каждой строки сумма двух вероятностей равна 1. Например, $0.9259 + 0.0741 = 1.0000$.

**Интуиция:** ответственности показывают, насколько каждое слово «тянется» к каждой теме при текущих параметрах. Например, слово «кошка» в $d_1$ почти полностью относится к теме 1 (0.9259), а слово «на» в $d_1$ — в основном к теме 2 (0.7143). Это соответствует нашим начальным представлениям о темах.

### 4.1.5.4 M-шаг: обновление параметров

На M-шаге мы пересчитываем параметры $P(w \mid z)$ и $P(z \mid d)$, используя ответственности, полученные на E-шаге.

#### Обновление $P(w \mid z)$

Формула:

$$
P(w \mid z) = \frac{\sum_{d=1}^{M} n(d,w) P(z \mid d,w)}{\sum_{d=1}^{M} \sum_{w' \in V} n(d,w') P(z \mid d,w')}.
$$

**Интерпретация:** числитель — это ожидаемое число раз, когда слово $w$ было порождено темой $z$. Знаменатель — ожидаемое общее число слов, порождённых темой $z$. Таким образом, $P(w \mid z)$ — это нормированная частота слова $w$ в теме $z$.

Вычислим числители для каждого слова.

| Слово | $\sum_d n(d,w) P(z{=}1 \mid d,w)$ | $\sum_d n(d,w) P(z{=}2 \mid d,w)$ |
|-------|-----------------------------------|-----------------------------------|
| кошка | $0.9259 + 0.9106 = 1.8365$ | $0.0741 + 0.0894 = 0.1635$ |
| сидит | $0.8571 + 0.6316 = 1.4887$ | $0.1429 + 0.3684 = 0.5113$ |
| на    | $0.2857 + 0.1026 + 0.2458 = 0.6341$ | $0.7143 + 0.8974 + 0.7542 = 2.3659$ |
| окне  | $0.1525$ | $0.8475$ |
| собака | $0.9146$ | $0.0854$ |
| крыльце | $0.0411$ | $0.9589$ |
| спит  | $0.9483$ | $0.0517$ |
| диване | $0.1401$ | $0.8599$ |

Суммы по темам:

$$
Z_1 = 1.8365 + 1.4887 + 0.6341 + 0.1525 + 0.9146 + 0.0411 + 0.9483 + 0.1401 = 6.1559,
$$

$$
Z_2 = 0.1635 + 0.5113 + 2.3659 + 0.8475 + 0.0854 + 0.9589 + 0.0517 + 0.8599 = 5.8441.
$$

**Проверка:** $Z_1 + Z_2 = 6.1559 + 5.8441 = 12.0000$, что равно общему числу словоупотреблений. Это подтверждает корректность вычислений: каждая ответственность «распределяется» между темами, и сумма ожидаемых counts по темам равна общему числу слов.

Новые вероятности:

**$P(w \mid z=1)$:**

| слово | вероятность |
|-------|-------------|
| кошка | $1.8365 / 6.1559 \approx 0.2983$ |
| сидит | $1.4887 / 6.1559 \approx 0.2418$ |
| на    | $0.6341 / 6.1559 \approx 0.1030$ |
| окне  | $0.1525 / 6.1559 \approx 0.0248$ |
| собака | $0.9146 / 6.1559 \approx 0.1486$ |
| крыльце | $0.0411 / 6.1559 \approx 0.0067$ |
| спит  | $0.9483 / 6.1559 \approx 0.1541$ |
| диване | $0.1401 / 6.1559 \approx 0.0228$ |

Проверка суммы:

$$
0.2983 + 0.2418 + 0.1030 + 0.0248 + 0.1486 + 0.0067 + 0.1541 + 0.0228 \approx 1.0001 \approx 1.
$$

**P(w \|  z=2):**

| слово | вероятность |
|-------|-------------|
| кошка | $0.1635 / 5.8441 \approx 0.0280$ |
| сидит | $0.5113 / 5.8441 \approx 0.0875$ |
| на    | $2.3659 / 5.8441 \approx 0.4048$ |
| окне  | $0.8475 / 5.8441 \approx 0.1450$ |
| собака | $0.0854 / 5.8441 \approx 0.0146$ |
| крыльце | $0.9589 / 5.8441 \approx 0.1641$ |
| спит  | $0.0517 / 5.8441 \approx 0.0088$ |
| диване | $0.8599 / 5.8441 \approx 0.1471$ |

Проверка суммы:

$$
0.0280 + 0.0875 + 0.4048 + 0.1450 + 0.0146 + 0.1641 + 0.0088 + 0.1471 \approx 0.9999 \approx 1.
$$

**Что изменилось?** После первой итерации тема 1 усилила слова «кошка» (0.25 → 0.2983), «сидит» (0.20 → 0.2418), «спит» (0.15 → 0.1541), а тема 2 усилила «на» (0.30 → 0.4048), «крыльце» (0.20 → 0.1641), «диване» (0.15 → 0.1471). Это отражает то, что слова «кошка», «сидит», «спит» чаще относятся к теме 1, а «на», «крыльце», «диване» — к теме 2.

#### Обновление $P(z \mid d)$

Формула:

$$
P(z \mid d) = \frac{\sum_{w \in V} n(d,w) P(z \mid d,w)}{\sum_{w \in V} n(d,w)}.
$$

**Интерпретация:** числитель — ожидаемое число слов в документе $d$, порождённых темой $z$. Знаменатель — общее число слов в документе (в нашем случае 4).

Для $d_1$:

$$
\sum_w n(d_1,w) P(z=1 \mid d_1,w) = 0.9259 + 0.8571 + 0.2857 + 0.1525 = 2.2212,
$$

$$
\sum_w n(d_1,w) P(z=2 \mid d_1,w) = 0.0741 + 0.1429 + 0.7143 + 0.8475 = 1.7788.
$$

$$
P(z=1 \mid d_1) = \frac{2.2212}{4} = 0.5553, \quad P(z=2 \mid d_1) = \frac{1.7788}{4} = 0.4447.
$$

Проверка: $0.5553 + 0.4447 = 1.0000$.

Для $d_2$:

$$
\sum_w n(d_2,w) P(z=1 \mid d_2,w) = 0.9146 + 0.6316 + 0.1026 + 0.0411 = 1.6899,
$$

$$
\sum_w n(d_2,w) P(z=2 \mid d_2,w) = 0.0854 + 0.3684 + 0.8974 + 0.9589 = 2.3101.
$$

$$
P(z=1 \mid d_2) = \frac{1.6899}{4} = 0.4225, \quad P(z=2 \mid d_2) = \frac{2.3101}{4} = 0.5775.
$$

Для $d_3$:

$$
\sum_w n(d_3,w) P(z=1 \mid d_3,w) = 0.9106 + 0.9483 + 0.2458 + 0.1401 = 2.2448,
$$

$$
\sum_w n(d_3,w) P(z=2 \mid d_3,w) = 0.0894 + 0.0517 + 0.7542 + 0.8599 = 1.7552.
$$

$$
P(z=1 \mid d_3) = \frac{2.2448}{4} = 0.5612, \quad P(z=2 \mid d_3) = \frac{1.7552}{4} = 0.4388.
$$

**Сводка после первой итерации:**

| Документ | P(z=1  \|  d) | P(z=2 \|  d) |
|----------|-----------------|-----------------|
| $d_1$ | 0.5553 | 0.4447 |
| $d_2$ | 0.4225 | 0.5775 |
| $d_3$ | 0.5612 | 0.4388 |

**Что изменилось?** Для $d_1$ вероятность темы 1 немного снизилась (0.6 → 0.5553), а темы 2 — выросла (0.4 → 0.4447). Для $d_2$ вероятность темы 1 выросла (0.3 → 0.4225), а темы 2 — снизилась (0.7 → 0.5775). Для $d_3$ изменения минимальны.

### 4.1.5.5 Вычисление логарифма правдоподобия

EM-алгоритм гарантирует, что логарифм правдоподобия не убывает на каждой итерации. Проверим это.

**Начальный логарифм правдоподобия** (до EM) с исходными параметрами:

Для каждой пары вычислим $P(w,d) = P(z{=}1|d)P(w|z{=}1) + P(z{=}2|d)P(w|z{=}2)$ и логарифм.

Для $d_1$:
- кошка: $0.6 \cdot 0.25 + 0.4 \cdot 0.03 = 0.162$, $\ln = -1.820$;
- сидит: $0.6 \cdot 0.20 + 0.4 \cdot 0.05 = 0.14$, $\ln = -1.966$;
- на: $0.6 \cdot 0.08 + 0.4 \cdot 0.30 = 0.168$, $\ln = -1.784$;
- окне: $0.6 \cdot 0.03 + 0.4 \cdot 0.25 = 0.118$, $\ln = -2.136$.

Сумма $d_1 = -7.706$.

Для $d_2$:
- собака: $0.3 \cdot 0.25 + 0.7 \cdot 0.01 = 0.082$, $\ln = -2.501$;
- сидит: $0.3 \cdot 0.20 + 0.7 \cdot 0.05 = 0.095$, $\ln = -2.354$;
- на: $0.3 \cdot 0.08 + 0.7 \cdot 0.30 = 0.234$, $\ln = -1.453$;
- крыльце: $0.3 \cdot 0.02 + 0.7 \cdot 0.20 = 0.146$, $\ln = -1.924$.

Сумма $d_2 = -8.232$.

Для $d_3$:
- кошка: $0.55 \cdot 0.25 + 0.45 \cdot 0.03 = 0.151$, $\ln = -1.890$;
- спит: $0.55 \cdot 0.15 + 0.45 \cdot 0.01 = 0.087$, $\ln = -2.442$;
- на: $0.55 \cdot 0.08 + 0.45 \cdot 0.30 = 0.179$, $\ln = -1.720$;
- диване: $0.55 \cdot 0.02 + 0.45 \cdot 0.15 = 0.0785$, $\ln = -2.544$.

Сумма $d_3 = -8.596$.

Общий начальный логарифм правдоподобия:

$$
\ell_0 = -7.706 -8.232 -8.596 = -24.534.
$$

**Логарифм правдоподобия после первой итерации** (с обновлёнными параметрами):

Используем новые $P(w|z)$ и $P(z|d)$, округлённые до 4 знаков.

Для $d_1$:
- кошка: $0.5553 \cdot 0.2983 + 0.4447 \cdot 0.0280 = 0.1781$, $\ln \approx -1.724$;
- сидит: $0.5553 \cdot 0.2418 + 0.4447 \cdot 0.0875 = 0.1732$, $\ln \approx -1.753$;
- на: $0.5553 \cdot 0.1030 + 0.4447 \cdot 0.4048 = 0.2372$, $\ln \approx -1.439$;
- окне: $0.5553 \cdot 0.0248 + 0.4447 \cdot 0.1450 = 0.0783$, $\ln \approx -2.547$.

Сумма $d_1 \approx -7.463$.

Для $d_2$:
- собака: $0.4225 \cdot 0.1486 + 0.5775 \cdot 0.0146 = 0.0712$, $\ln \approx -2.642$;
- сидит: $0.4225 \cdot 0.2418 + 0.5775 \cdot 0.0875 = 0.1527$, $\ln \approx -1.879$;
- на: $0.4225 \cdot 0.1030 + 0.5775 \cdot 0.4048 = 0.2773$, $\ln \approx -1.283$;
- крыльце: $0.4225 \cdot 0.0067 + 0.5775 \cdot 0.1641 = 0.0976$, $\ln \approx -2.327$.

Сумма $d_2 \approx -8.131$.

Для $d_3$:
- кошка: $0.5612 \cdot 0.2983 + 0.4388 \cdot 0.0280 = 0.1797$, $\ln \approx -1.716$;
- спит: $0.5612 \cdot 0.1541 + 0.4388 \cdot 0.0088 = 0.0904$, $\ln \approx -2.404$;
- на: $0.5612 \cdot 0.1030 + 0.4388 \cdot 0.4048 = 0.2355$, $\ln \approx -1.446$;
- диване: $0.5612 \cdot 0.0228 + 0.4388 \cdot 0.1471 = 0.0774$, $\ln \approx -2.559$.

Сумма $d_3 \approx -8.125$.

Общий логарифм правдоподобия после первой итерации:

$$
\ell_1 = -7.463 -8.131 -8.125 = -23.719.
$$

**Проверка:** $\ell_1 > \ell_0$ ($-23.719 > -24.534$). Правдоподобие возросло, что подтверждает корректность шага EM.

### 4.1.5.6 Итерация 2 и последующие

На второй итерации повторяются E-шаг с обновлёнными параметрами и M-шаг. Процесс сходится за несколько итераций. Приведём финальные параметры после 10 итераций (округлённо до 3 знаков).

**Итоговое $P(w|z=1)$:**

| кошка | сидит | на | окне | собака | крыльце | спит | диване |
|-------|-------|----|------|--------|---------|------|--------|
| 0.258 | 0.207 | 0.088 | 0.021 | 0.240 | 0.007 | 0.156 | 0.023 |

**Итоговое $P(w|z=2)$:**

| кошка | сидит | на | окне | собака | крыльце | спит | диване |
|-------|-------|----|------|--------|---------|------|--------|
| 0.021 | 0.043 | 0.421 | 0.156 | 0.002 | 0.177 | 0.009 | 0.171 |

**Итоговое $P(z|d)$:**

| Документ | P(z=1\| d) | P(z=2 \| d) |
|----------|-------------|-------------|
| $d_1$ | 0.613 | 0.387 |
| $d_2$ | 0.405 | 0.595 |
| $d_3$ | 0.582 | 0.418 |

Логарифм правдоподобия после 10 итераций: $\ell \approx -22.9$, что показывает дальнейший рост по сравнению с $\ell_1 = -23.719$.

### 4.1.5.7 Итоговая матрица $P(w \mid d)$

После обучения модели мы можем вычислить вероятность каждого слова в каждом документе по формуле:

$$
P(w \mid d) = \sum_{z=1}^{K} P(w \mid z) P(z \mid d).
$$

Эта матрица показывает, как модель «объясняет» наблюдаемые документы. Приведём её для всех документов (округлённо до 4 знаков).

| Слово | $P(w \mid d_1)$ | $P(w \mid d_2)$ | $P(w \mid d_3)$ |
|-------|-----------------|-----------------|-----------------|
| кошка | $0.613 \cdot 0.258 + 0.387 \cdot 0.021 = 0.1663$ | $0.405 \cdot 0.258 + 0.595 \cdot 0.021 = 0.1170$ | $0.582 \cdot 0.258 + 0.418 \cdot 0.021 = 0.1589$ |
| сидит | $0.613 \cdot 0.207 + 0.387 \cdot 0.043 = 0.1435$ | $0.405 \cdot 0.207 + 0.595 \cdot 0.043 = 0.1094$ | $0.582 \cdot 0.207 + 0.418 \cdot 0.043 = 0.1384$ |
| на    | $0.613 \cdot 0.088 + 0.387 \cdot 0.421 = 0.2169$ | $0.405 \cdot 0.088 + 0.595 \cdot 0.421 = 0.2861$ | $0.582 \cdot 0.088 + 0.418 \cdot 0.421 = 0.2272$ |
| окне  | $0.613 \cdot 0.021 + 0.387 \cdot 0.156 = 0.0732$ | $0.405 \cdot 0.021 + 0.595 \cdot 0.156 = 0.1013$ | $0.582 \cdot 0.021 + 0.418 \cdot 0.156 = 0.0774$ |
| собака | $0.613 \cdot 0.240 + 0.387 \cdot 0.002 = 0.1479$ | $0.405 \cdot 0.240 + 0.595 \cdot 0.002 = 0.0984$ | $0.582 \cdot 0.240 + 0.418 \cdot 0.002 = 0.1405$ |
| крыльце | $0.613 \cdot 0.007 + 0.387 \cdot 0.177 = 0.0728$ | $0.405 \cdot 0.007 + 0.595 \cdot 0.177 = 0.1082$ | $0.582 \cdot 0.007 + 0.418 \cdot 0.177 = 0.0781$ |
| спит  | $0.613 \cdot 0.156 + 0.387 \cdot 0.009 = 0.0991$ | $0.405 \cdot 0.156 + 0.595 \cdot 0.009 = 0.0685$ | $0.582 \cdot 0.156 + 0.418 \cdot 0.009 = 0.0946$ |
| диване | $0.613 \cdot 0.023 + 0.387 \cdot 0.171 = 0.0803$ | $0.405 \cdot 0.023 + 0.595 \cdot 0.171 = 0.1111$ | $0.582 \cdot 0.023 + 0.418 \cdot 0.171 = 0.0849$ |

**Проверка:** сумма по каждому столбцу должна быть равна 1. Проверим для $d_1$:

$$
0.1663 + 0.1435 + 0.2169 + 0.0732 + 0.1479 + 0.0728 + 0.0991 + 0.0803 = 1.0000.
$$

Аналогично для $d_2$ и $d_3$ суммы равны 1 (с учётом округления).

**Как читать эту матрицу?** Она показывает, с какой вероятностью модель ожидает увидеть каждое слово в каждом документе. Например, в $d_1$ наиболее вероятны слова «на» (0.2169), «кошка» (0.1663) и «собака» (0.1479). Это отражает то, что тема 1, доминирующая в $d_1$ (0.613), включает как «кошку», так и «собаку» (животные), а тема 2 добавляет предлог «на». В $d_2$ максимальна вероятность слова «на» (0.2861), затем «кошка» (0.1170) и «диване» (0.1111), что связано с доминированием темы 2 (0.595).

**Важно:** матрица $P(w \mid d)$ — это не исходные частоты, а **реконструкция** модели. Она показывает, какие слова модель считает типичными для документа, исходя из выученных тем. Сравнивая её с исходной матрицей $n(d,w)$, можно оценить качество модели.

---

## 4.1.6 Интерпретация результатов

### 4.1.6.1 Интерпретация тем

Полученные распределения слов по темам имеют ясную интерпретацию.

**Тема 1 ($z=1$)** выделяет слова, связанные с животными и действиями:

| слово | P(w \|  z=1) |
|-------|-----------------|
| кошка | 0.258 |
| собака | 0.240 |
| сидит | 0.207 |
| спит | 0.156 |
| на | 0.088 |
| диване | 0.023 |
| окне | 0.021 |
| крыльце | 0.007 |

Эту тему можно условно назвать **«животные и действия»**.

**Тема 2 ($z=2$)** акцентирует места и предлоги:

| слово | P(w \|  z=2) |
|-------|-----------------|
| на | 0.421 |
| крыльце | 0.177 |
| диване | 0.171 |
| окне | 0.156 |
| сидит | 0.043 |
| кошка | 0.021 |
| спит | 0.009 |
| собака | 0.002 |

Эту тему можно условно назвать **«места и предлоги»**.

**Ключевое наблюдение:** pLSA **автоматически** разделила лексику на две смысловые группы, хотя ей не было дано никаких явных указаний. Модель не знает значений слов — она видит только совместную встречаемость. Но статистика такова, что «кошка» и «собака» чаще встречаются с «сидит», а «окне», «крыльце», «диване» — с предлогом «на». Это и привело к разделению.

### 4.1.6.2 Интерпретация документов

Распределения тем по документам показывают, как каждый документ распределён по темам:

| Документ | $P(z=1 \mid d)$ | $P(z=2 \mid d)$ | Доминирующая тема |
|----------|-----------------|-----------------|-------------------|
| $d_1$ | 0.613 | 0.387 | Тема 1 (животные) |
| $d_2$ | 0.405 | 0.595 | Тема 2 (места) |
| $d_3$ | 0.582 | 0.418 | Тема 1 (животные) |

**Интерпретация:**

- Документ 1 («кошка сидит на окне») и документ 3 («кошка спит на диване») имеют более высокую вероятность темы 1 (0.613 и 0.582). Это логично: оба документа содержат слова «кошка», «сидит»/«спит», которые относятся к теме 1.

- Документ 2 («собака сидит на крыльце») имеет более высокую вероятность темы 2 (0.595). Это тоже логично: в нём есть «крыльце» (0.177 в теме 2) и «на» (0.421 в теме 2), которые сильнее связаны с темой 2.

**Тонкий момент:** документ 2 содержит слово «сидит», которое в теме 1 имеет высокую вероятность (0.207). Однако из-за слов «крыльце» и «на» документ в целом смещается к теме 2. Это показывает, что pLSA учитывает **все** слова документа, а не только доминирующие.

### 4.1.6.3 Сравнение с исходными данными

Сравним исходную матрицу $n(d,w)$ с реконструкцией $P(w \mid d)$:

| Слово | $n(d_1,w)$ | $P(w \mid d_1)$ | $n(d_2,w)$ | $P(w \mid d_2)$ | $n(d_3,w)$ | $P(w \mid d_3)$ |
|-------|------------|-----------------|------------|-----------------|------------|-----------------|
| кошка | 1 | 0.1663 | 0 | 0.1170 | 1 | 0.1589 |
| сидит | 1 | 0.1435 | 1 | 0.1094 | 0 | 0.1384 |
| на    | 1 | 0.2169 | 1 | 0.2861 | 1 | 0.2272 |
| окне  | 1 | 0.0732 | 0 | 0.1013 | 0 | 0.0774 |
| собака | 0 | 0.1479 | 1 | 0.0984 | 0 | 0.1405 |
| крыльце | 0 | 0.0728 | 1 | 0.1082 | 0 | 0.0781 |
| спит  | 0 | 0.0991 | 0 | 0.0685 | 1 | 0.0946 |
| диване | 0 | 0.0803 | 0 | 0.1111 | 1 | 0.0849 |

**Наблюдения:**

1. Модель **сглаживает** данные: даже если слово не встречалось в документе (например, «собака» в $d_1$), модель всё равно даёт ему ненулевую вероятность (0.1479), потому что тема 1, доминирующая в $d_1$, включает «собаку». Это свойство вероятностных моделей: они обобщают, а не просто запоминают.

2. Слова, которые действительно встречаются в документе, обычно имеют более высокие вероятности. Например, «на» в $d_1$ имеет $P(w \mid d_1) = 0.2169$, что выше среднего.

3. Однако из-за сглаживания некоторые не встречающиеся слова могут получить высокие вероятности. Например, «собака» в $d_1$ (0.1479) выше, чем «окне» (0.0732), хотя «окне» встречается в $d_1$, а «собака» — нет. Это происходит потому, что тема 1 сильно связана с «собакой», и документ $d_1$ в основном относится к теме 1.

**Это не ошибка модели, а её свойство:** pLSA ищет латентные темы, которые объясняют данные, и если тема 1 включает и «кошку», и «собаку», то документ про кошку будет иметь высокую вероятность и для «собаки».

### 4.1.6.4 Ограничения pLSA

Несмотря на интерпретируемость и статистическую строгость, pLSA имеет два серьёзных недостатка:

1. **Число параметров $\theta$ растёт линейно с числом документов.**  
   Для каждого документа обучается свой вектор $P(z \mid d)$. При большом $M$ это приводит к переобучению.

2. **Модель не определяет вероятностного распределения для новых документов.**  
   Если появляется документ, не входивший в обучающую выборку, для него нет параметра $P(z \mid d_{\text{new}})$, и его нужно оценивать заново, удерживая темы фиксированными.

Эти ограничения преодолеваются в модели **латентного размещения Дирихле** (LDA), которая добавляет байесовские априорные распределения на параметры и тем самым решает проблему переобучения и обеспечивает естественный способ обработки новых документов. Переход к LDA будет рассмотрен в следующей части.



# Часть 4.2. Латентное размещение Дирихле (LDA)

## 4.2.1 Байесовское расширение pLSA

### Что не так с pLSA?

Вероятностный латентный семантический анализ, рассмотренный в предыдущей части, позволил моделировать документы как смеси скрытых тем. Однако у него есть два принципиальных ограничения.

**Первое ограничение: число параметров растёт с числом документов.**  
В pLSA для каждого документа $d$ обучается свой вектор $\theta_d = (P(z=1 \mid d), \ldots, P(z=K \mid d))$. Общее число параметров $\theta$ равно $M \times K$, где $M$ — число документов. Если коллекция содержит миллион документов, а тем $K=100$, то только для $\theta$ нужно хранить $10^8$ параметров. Это ведёт к переобучению: модель «запоминает» каждый документ, вместо того чтобы выучить общие закономерности.

**Второе ограничение: нет вероятностного механизма для новых документов.**  
Если после обучения pLSA появляется новый документ, не входивший в обучающую выборку, для него нет параметра $P(z \mid d_{\text{new}})$. Его нужно оценивать отдельно, удерживая темы $\phi_z$ фиксированными. Это неудобно и не вполне байесовски: мы не можем сказать, «какова вероятность нового документа при данной модели».

### Как LDA решает эти проблемы?

Модель **латентного размещения Дирихле** (Latent Dirichlet Allocation, LDA), предложенная Блеем, Нг и Джорданом в 2003 году, превращает pLSA в **полностью байесовскую модель**. Идея:

- Вместо того чтобы считать $\theta_d$ и $\phi_z$ **фиксированными параметрами**, LDA рассматривает их как **случайные векторы**, порождённые из распределений Дирихле.
- Априорные распределения Дирихле имеют параметры $\alpha$ и $\beta$, которые **общие для всех документов и тем**. Это означает, что число глобальных параметров не растёт с числом документов.
- Новый документ можно обработать, вычислив апостериорное распределение $\theta_{\text{new}}$ при фиксированных темах $\phi_z$. Это делается через байесовский вывод.

**Интуиция:** в pLSA мы считаем, что у каждого документа свой уникальный «рецепт» смеси тем, и мы оцениваем его независимо. В LDA мы считаем, что все документы черпают свои «рецепты» из общего распределения Дирихле. Это сглаживает оценки и позволяет обобщать.

### Что такое байесовский подход?

В частотной статистике (MLE) параметры — это неизвестные константы, которые мы оцениваем по данным. В байесовском подходе параметры — это **случайные величины** с априорным распределением. Мы обновляем это распределение по мере наблюдения данных, получая апостериорное распределение.

Формально, теорема Байеса:

$$
P(\text{параметры} \mid \text{данные}) = \frac{P(\text{данные} \mid \text{параметры}) P(\text{параметры})}{P(\text{данные})}.
$$

- $P(\text{параметры})$ — априорное распределение (до наблюдения данных);
- $P(\text{данные} \mid \text{параметры})$ — правдоподобие;
- $P(\text{параметры} \mid \text{данные})$ — апостериорное распределение (после наблюдения данных).

В LDA мы выбираем априорные распределения так, чтобы они были **сопряжёнными** с правдоподобием. Это означает, что апостериорное распределение имеет ту же функциональную форму, что и априорное, что упрощает вычисления.

---

## 4.2.2 Распределение Дирихле

### Что такое симплекс?

Прежде чем определить распределение Дирихле, вспомним, что такое **симплекс**. Вектор $\theta = (\theta_1, \theta_2, \ldots, \theta_K)$ лежит в $(K-1)$-мерном симплексе, если:

$$
\theta_i \ge 0 \quad \text{для всех } i, \qquad \sum_{i=1}^K \theta_i = 1.
$$

Например, при $K=2$ симплекс — это отрезок прямой от $(1,0)$ до $(0,1)$. При $K=3$ — это треугольник с вершинами $(1,0,0)$, $(0,1,0)$, $(0,0,1)$. При $K>3$ — это многомерный аналог треугольника.

**Почему это важно?** Потому что $\theta_d$ — это распределение вероятностей тем, и оно должно лежать на симплексе. Аналогично $\phi_z$ — распределение вероятностей слов, тоже на симплексе.

### Определение распределения Дирихле

Распределение Дирихле — это распределение на симплексе. Оно задаётся параметром $\alpha = (\alpha_1, \ldots, \alpha_K)$, где $\alpha_i > 0$. Плотность:

$$
p(\theta \mid \alpha) = \frac{\Gamma\left(\sum_{i=1}^K \alpha_i\right)}{\prod_{i=1}^K \Gamma(\alpha_i)} \prod_{i=1}^K \theta_i^{\alpha_i - 1},
$$

где $\Gamma$ — гамма-функция, обобщение факториала:

$$
\Gamma(x) = \int_0^\infty t^{x-1} e^{-t} \, dt, \qquad \Gamma(n) = (n-1)! \text{ для целых } n.
$$

Множитель перед произведением — это нормировочная константа, обеспечивающая, что интеграл по симплексу равен 1.

### Свойства распределения Дирихле

1. **Среднее значение:**
   $$
   \mathbb{E}[\theta_i] = \frac{\alpha_i}{\sum_{j=1}^K \alpha_j}.
   $$
   Если все $\alpha_i$ равны $\alpha$, то среднее — это равномерное распределение $1/K$.

2. **Дисперсия:**
   $$
   \mathrm{Var}[\theta_i] = \frac{\mathbb{E}[\theta_i](1 - \mathbb{E}[\theta_i])}{\alpha_0 + 1}, \qquad \alpha_0 = \sum_{j=1}^K \alpha_j.
   $$
   Чем больше $\alpha_0$, тем меньше дисперсия, то есть распределение концентрируется вокруг среднего.

3. **Влияние параметра $\alpha$:**
   - При $\alpha_i < 1$ распределение концентрируется на **разреженных** векторах: большинство компонент близки к нулю, одна или несколько доминируют. Это соответствует ситуации, когда документ посвящён небольшому числу тем.
   - При $\alpha_i = 1$ распределение **равномерное** на симплексе.
   - При $\alpha_i > 1$ распределение концентрируется вокруг **равномерного** вектора: все компоненты близки к $1/K$.

**Интуиция:** параметр $\alpha$ управляет «уверенностью» априорного распределения. Малые $\alpha$ говорят: «Я ожидаю, что документ будет смесью небольшого числа тем». Большие $\alpha$ говорят: «Я ожидаю, что документ будет равномерно смешан из всех тем».

### Симметричное распределение Дирихле

В LDA обычно используют **симметричное** распределение Дирихле, где все $\alpha_i$ равны одному скаляру $\alpha$. Тогда:

$$
p(\theta \mid \alpha) = \frac{\Gamma(K\alpha)}{\Gamma(\alpha)^K} \prod_{i=1}^K \theta_i^{\alpha - 1}.
$$

Аналогично, распределение слов в теме $\phi_z$ имеет априорное распределение Дирихле с симметричным параметром $\beta$:

$$
p(\phi_z \mid \beta) = \frac{\Gamma(N\beta)}{\Gamma(\beta)^N} \prod_{w=1}^N \phi_{zw}^{\beta - 1},
$$

где $N$ — размер словаря.

### Сопряжённость с мультиномиальным распределением

**Ключевое свойство:** распределение Дирихле является **сопряжённым** к мультиномиальному распределению. Это означает, что если априорное распределение вероятностей мультиномиальной модели — Дирихле, то апостериорное распределение тоже Дирихле.

Формально, если:

$$
\theta \sim \text{Dirichlet}(\alpha),
$$

и мы наблюдаем counts $n = (n_1, \ldots, n_K)$ из мультиномиального распределения с параметром $\theta$, то:

$$
\theta \mid n \sim \text{Dirichlet}(\alpha + n) = \text{Dirichlet}(\alpha_1 + n_1, \ldots, \alpha_K + n_K).
$$

**Почему это важно?** Именно благодаря сопряжённости мы можем аналитически проинтегрировать $\theta$ и $\phi$ в LDA и получить простое условное распределение для сэмплирования Гиббса. Без сопряжённости нам пришлось бы использовать численные методы для каждого шага, что было бы вычислительно неэффективно.

**Вывод сопряжённости:**  
Апостериорное распределение:

$$
p(\theta \mid n) \propto p(n \mid \theta) p(\theta \mid \alpha) \propto \prod_{i=1}^K \theta_i^{n_i} \cdot \prod_{i=1}^K \theta_i^{\alpha_i - 1} = \prod_{i=1}^K \theta_i^{\alpha_i + n_i - 1}.
$$

Это в точности форма распределения Дирихле с параметром $\alpha + n$. Нормировочная константа автоматически подстраивается.

---

## 4.2.3 Порождающий процесс LDA

### Формальное описание

Порождающий процесс для коллекции из $M$ документов и словаря из $N$ слов выглядит следующим образом.

**Шаг 1: Порождение тем (глобально).**  
Для каждой темы $z = 1, \ldots, K$ выбрать распределение слов:

$$
\phi_z \sim \text{Dirichlet}(\beta).
$$

Это означает, что каждая тема — это распределение вероятностей по всем $N$ словам словаря. Параметр $\beta$ общий для всех тем.

**Шаг 2: Порождение документов.**  
Для каждого документа $d = 1, \ldots, M$:

1. Выбрать распределение тем:
   $$
   \theta_d \sim \text{Dirichlet}(\alpha).
   $$
   Это вектор длины $K$, показывающий, какая доля слов документа относится к каждой теме.

2. Для каждой позиции слова $n = 1, \ldots, N_d$:
   - Выбрать тему:
     $$
     z_{d,n} \sim \text{Multinomial}(\theta_d).
     $$
     Это означает, что тема выбирается случайно с вероятностями $\theta_d$.
   - Выбрать слово:
     $$
     w_{d,n} \sim \text{Multinomial}(\phi_{z_{d,n}}).
     $$
     Это означает, что слово выбирается случайно из распределения, соответствующего выбранной теме.

### Схема порождающего процесса

$$
\underbrace{\alpha}_{\text{гиперпараметр}} \longrightarrow \underbrace{\theta_d}_{\text{темы документа}} \longrightarrow \underbrace{z_{d,n}}_{\text{тема слова}} \longrightarrow \underbrace{w_{d,n}}_{\text{слово}},
$$

$$
\underbrace{\beta}_{\text{гиперпараметр}} \longrightarrow \underbrace{\phi_z}_{\text{слова темы}} \longrightarrow \underbrace{w_{d,n}}_{\text{слово}}.
$$

### Интуиция

Представьте, что вы пишете документ. У вас есть:

- **Общий словарь тем** (например, «спорт», «политика», «наука»), и для каждой темы — свой набор характерных слов.
- **Личный рецепт документа** $\theta_d$: например, 70% спорта, 20% политики, 10% науки.
- Вы пишете слово за словом: сначала бросаете «тематическую монетку» с вероятностями $\theta_d$, затем, в зависимости от выпавшей темы, бросаете «словесную монетку» с вероятностями $\phi_z$.

**Важно:** мы не утверждаем, что реальный автор текста действительно так делает. Это математическая модель, которая, как мы надеемся, улавливает статистические закономерности языка.

### Отличие от pLSA

В pLSA:

- $\theta_d$ и $\phi_z$ — **фиксированные параметры**, которые мы оцениваем через EM.
- Число параметров $\theta$ растёт с числом документов.

В LDA:

- $\theta_d$ и $\phi_z$ — **случайные величины**, порождённые из распределений Дирихле.
- Число **глобальных** параметров — только $\alpha$ и $\beta$ (гиперпараметры), которые не растут с числом документов.
- Каждый документ имеет свою $\theta_d$, но она не «хранится» как параметр — она интегрируется.

### Гиперпараметры $\alpha$ и $\beta$

Гиперпараметры $\alpha$ и $\beta$ задаются пользователем. Обычно выбирают малые значения:

$$
\alpha = 0.1, \qquad \beta = 0.01.
$$

**Почему малые?**

- Малые $\alpha$ означают, что априорное распределение тем в документе концентрируется на разреженных векторах: мы ожидаем, что каждый документ посвящён небольшому числу тем, а не равномерно смешан из всех.
- Малые $\beta$ означают, что априорное распределение слов в теме концентрируется на разреженных векторах: мы ожидаем, что каждая тема использует небольшой набор характерных слов, а не все слова словаря.

**Тонкий момент:** выбор $\alpha$ и $\beta$ влияет на результат. Слишком большие значения приводят к «размытым» темам, слишком малые — к вырождению (тема может «схлопнуться» на одно слово). На практике значения подбирают экспериментально или используют эмпирические рекомендации.

---

## 4.2.4 Совместное распределение и задача вывода

### Совместное распределение

Совместное распределение всех наблюдаемых слов $W = \{w_{d,n}\}$, скрытых тем $Z = \{z_{d,n}\}$, параметров $\Theta = \{\theta_d\}$ и $\Phi = \{\phi_z\}$ имеет вид:

$$
P(W, Z, \Theta, \Phi \mid \alpha, \beta) = \prod_{z=1}^K P(\phi_z \mid \beta) \prod_{d=1}^M P(\theta_d \mid \alpha) \prod_{n=1}^{N_d} P(z_{d,n} \mid \theta_d) P(w_{d,n} \mid \phi_{z_{d,n}}).
$$

**Разберём эту формулу по частям:**

- $\prod_{z=1}^K P(\phi_z \mid \beta)$ — априорное распределение тем (слов);
- $\prod_{d=1}^M P(\theta_d \mid \alpha)$ — априорное распределение тем в документах;
- $\prod_{n=1}^{N_d} P(z_{d,n} \mid \theta_d)$ — вероятность выбора темы для каждого слова;
- $\prod_{n=1}^{N_d} P(w_{d,n} \mid \phi_{z_{d,n}})$ — вероятность выбора слова при известной теме.

### Задача вывода

Цель вывода — вычислить **апостериорное распределение** скрытых переменных:

$$
P(Z, \Theta, \Phi \mid W, \alpha, \beta) = \frac{P(W, Z, \Theta, \Phi \mid \alpha, \beta)}{P(W \mid \alpha, \beta)}.
$$

**Проблема:** знаменатель $P(W \mid \alpha, \beta)$ требует интегрирования по всем возможным $\Theta$ и $\Phi$:

$$
P(W \mid \alpha, \beta) = \int \int \sum_Z P(W, Z, \Theta, \Phi \mid \alpha, \beta) \, d\Theta \, d\Phi.
$$

Этот интеграл **не имеет аналитической формы** из-за связи между $\theta$ и $\phi$ через темы. Именно поэтому применяют приближённые методы.

### Два основных подхода

1. **Сэмплирование Гиббса** — метод Монте-Карло с марковскими цепями (MCMC). Мы итеративно сэмплируем каждую скрытую переменную из её условного распределения. Это интуитивно понятно и широко используется.

2. **Вариационный вывод** — метод оптимизации, в котором мы приближаем апостериорное распределение параметрическим семейством и минимизируем расхождение Кульбака — Лейблера. Это быстрее, но сложнее в реализации.

Мы сосредоточимся на сэмплировании Гиббса.

---

## 4.2.5 Сэмплирование Гиббса для LDA

### Идея сэмплирования Гиббса

Сэмплирование Гиббса — это метод Монте-Карло с марковскими цепями. Идея:

- Мы не вычисляем апостериорное распределение напрямую (это невозможно).
- Вместо этого мы строим марковскую цепь, стационарное распределение которой совпадает с апостериорным.
- На каждом шаге мы сэмплируем одну переменную из её условного распределения при фиксированных остальных.

В LDA мы **не храним $\theta$ и $\phi$ явно**. Вместо этого мы работаем только с назначениями тем $Z = \{z_{d,n}\}$ для каждого слова. Параметры $\Theta$ и $\Phi$ **интегрируются аналитически** благодаря сопряжённости Дирихле и мультиномиального распределения.

### Почему можно интегрировать $\Theta$ и $\Phi$?

Рассмотрим апостериорное распределение тем $Z$ при наблюдаемых словах $W$:

$$
P(Z \mid W, \alpha, \beta) = \frac{P(W, Z \mid \alpha, \beta)}{P(W \mid \alpha, \beta)}.
$$

Числитель можно вычислить, интегрируя по $\Theta$ и $\Phi$:

$$
P(W, Z \mid \alpha, \beta) = \int \int P(W, Z, \Theta, \Phi \mid \alpha, \beta) \, d\Theta \, d\Phi.
$$

Благодаря сопряжённости эти интегралы вычисляются аналитически. Результат:

$$
P(W, Z \mid \alpha, \beta) = \prod_{d=1}^M \frac{\Gamma\left(\sum_k \alpha_k\right)}{\prod_k \Gamma(\alpha_k)} \frac{\prod_k \Gamma(n_{d,k} + \alpha_k)}{\Gamma\left(\sum_k (n_{d,k} + \alpha_k)\right)} \cdot \prod_{k=1}^K \frac{\Gamma\left(\sum_w \beta_w\right)}{\prod_w \Gamma(\beta_w)} \frac{\prod_w \Gamma(n_{k,w} + \beta_w)}{\Gamma\left(\sum_w (n_{k,w} + \beta_w)\right)},
$$

где:

- $n_{d,k}$ — число раз, когда слово в документе $d$ было отнесено к теме $k$;
- $n_{k,w}$ — число раз, когда слово $w$ было отнесено к теме $k$ во всей коллекции.

**Тонкий момент:** это выражение получено путём интегрирования по $\Theta$ и $\Phi$. Мы не приводим полный вывод, но идея в том, что каждый интеграл имеет форму бета-функции, которая выражается через гамма-функции.

### Вывод условного распределения для одной темы

Нас интересует условное распределение темы одного слова $z_{d,n}$ при фиксированных всех остальных темах $Z^{-(d,n)}$ и наблюдаемых словах $W$:

$$
P(z_{d,n} = k \mid Z^{-(d,n)}, W, \alpha, \beta).
$$

Используя формулу Байеса и свойства сопряжённости, можно показать, что:

$$
P(z_{d,n} = k \mid Z^{-(d,n)}, W, \alpha, \beta) \propto \frac{n_{d,k}^{-(d,n)} + \alpha_k}{\sum_{k'} \left( n_{d,k'}^{-(d,n)} + \alpha_{k'} \right)} \cdot \frac{n_{k,w}^{-(d,n)} + \beta_w}{\sum_{w'} \left( n_{k,w'}^{-(d,n)} + \beta_{w'} \right)}.
$$

**Обозначения:**

- $n_{d,k}^{-(d,n)}$ — число слов в документе $d$, отнесённых к теме $k$, **не считая** текущее слово $(d,n)$;
- $n_{k,w}^{-(d,n)}$ — число раз, когда слово $w$ было связано с темой $k$ во всей коллекции, **не считая** текущее вхождение.

**Вывод (кратко):**  
Вероятность $P(z_{d,n} = k \mid \ldots)$ пропорциональна произведению двух множителей:

1. **Первый множитель** — вероятность темы $k$ в документе $d$:
   $$
   \frac{n_{d,k}^{-(d,n)} + \alpha_k}{\sum_{k'} \left( n_{d,k'}^{-(d,n)} + \alpha_{k'} \right)}.
   $$
   Это апостериорное среднее $\theta_{d,k}$ с учётом априорного параметра $\alpha_k$.

2. **Второй множитель** — вероятность слова $w$ в теме $k$:
   $$
   \frac{n_{k,w}^{-(d,n)} + \beta_w}{\sum_{w'} \left( n_{k,w'}^{-(d,n)} + \beta_{w'} \right)}.
   $$
   Это апостериорное среднее $\phi_{k,w}$ с учётом априорного параметра $\beta_w$.

### Упрощение для симметричных $\alpha$ и $\beta$

При использовании симметричных скалярных $\alpha$ и $\beta$ (одинаковых для всех тем и слов) формула упрощается:

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \frac{n_{k,w}^{-(d,n)} + \beta}{\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta},
$$

где $N$ — размер словаря.

**Тонкий момент:** знаменатель во втором сомножителе $\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta$ **зависит от $k$** — это общее число слов, отнесённых к теме $k$, плюс $N\beta$. Поэтому его **нельзя** опустить при сравнении тем для данного слова, если мы хотим получить точные вероятности. Однако на практике часто используют пропорциональность:

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \left( n_{k,w}^{-(d,n)} + \beta \right),
$$

а затем нормализуют. Это эквивалентно, если мы сравниваем темы для одного и того же слова $w$: знаменатель $\sum_{w'} n_{k,w'}^{-(d,n)} + N\beta$ зависит от $k$, но при нормализации по $k$ он всё равно учитывается. Однако если мы хотим точные вероятности, лучше использовать полную формулу.

**Практический совет:** в реализациях сэмплирования Гиббса обычно вычисляют:

$$
p_k = \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \frac{n_{k,w}^{-(d,n)} + \beta}{\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta},
$$

затем нормализуют $p_k$ и сэмплируют тему пропорционально $p_k$.

### Алгоритм сэмплирования Гиббса

1. **Инициализация:** случайно присвоить каждому слову в каждом документе тему $z_{d,n} \in \{1, \ldots, K\}$.
2. **Подсчёт счётчиков:** вычислить $n_{d,k}$ и $n_{k,w}$ для всех $d, k, w$.
3. **Итерации:** для каждой итерации $t = 1, \ldots, T$:
   - Для каждого документа $d$ и каждой позиции слова $n$:
     * Исключить текущее слово из счётчиков: $n_{d,z_{d,n}} \mathrel{-}= 1$, $n_{z_{d,n},w_{d,n}} \mathrel{-}= 1$.
     * Вычислить вероятности для всех $k = 1, \ldots, K$:
       $$
       p_k \propto \left( n_{d,k} + \alpha \right) \cdot \frac{n_{k,w} + \beta}{\sum_{w'} n_{k,w'} + N \beta}.
       $$
     * Нормализовать $p_k$ и сэмплировать новую тему $z_{d,n} \sim \text{Multinomial}(p)$.
     * Обновить счётчики: $n_{d,z_{d,n}} \mathrel{+}= 1$, $n_{z_{d,n},w_{d,n}} \mathrel{+}= 1$.
4. **Burn-in:** первые несколько сотен или тысяч итераций отбрасываются, так как цепь ещё не стабилизировалась.
5. **Сбор статистики:** после burn-in собираются сэмплы тем, по которым оцениваются $\theta$ и $\phi$.

### Оценка параметров $\theta$ и $\phi$

После достаточного числа итераций (burn-in) сэмплы тем стабилизируются, и можно оценить параметры:

$$
\hat{\theta}_{d,k} = \frac{n_{d,k} + \alpha}{\sum_{k'} (n_{d,k'} + \alpha)},
$$

$$
\hat{\phi}_{k,w} = \frac{n_{k,w} + \beta}{\sum_{w'} (n_{k,w'} + \beta)}.
$$

**Интерпретация:**

- $\hat{\theta}_{d,k}$ — оценка вероятности темы $k$ в документе $d$. Числитель — число слов документа $d$, отнесённых к теме $k$, плюс $\alpha$. Знаменатель — общее число слов документа плюс $K\alpha$.
- $\hat{\phi}_{k,w}$ — оценка вероятности слова $w$ в теме $k$. Числитель — число раз, когда слово $w$ было отнесено к теме $k$, плюс $\beta$. Знаменатель — общее число слов, отнесённых к теме $k$, плюс $N\beta$.

**Тонкий момент:** эти оценки — байесовские, с сглаживанием. Даже если слово $w$ ни разу не встречалось с темой $k$, его вероятность не равна нулю: $\hat{\phi}_{k,w} = \beta / (\sum_{w'} n_{k,w'} + N\beta) > 0$. Это решает проблему нулевых вероятностей и переобучения.

### Тонкие моменты сэмплирования Гиббса

1. **Сходимость.**  
   Цепь Маркова сходится к стационарному распределению (апостериорному) при $T \to \infty$. На практике нужно достаточно итераций для burn-in. Диагностика: отслеживание логарифма правдоподобия или тематической когерентности.

2. **Автокорреляция.**  
   Соседние сэмплы коррелированы. Чтобы получить независимые оценки, можно делать «прореживание» (thinning): сохранять каждый $m$-й сэмпл.

3. **Метки тем.**  
   Как и в pLSA, метки тем не идентифицируемы: перестановка тем не меняет правдоподобие. Это не проблема для интерпретации, но важно помнить при сравнении запусков.

4. **Выбор $K$, $\alpha$, $\beta$.**  
   $K$ — гиперпараметр, выбираемый по метрикам (перплексия, когерентность). $\alpha$ и $\beta$ обычно малы (0.1 и 0.01), но их можно подбирать.

5. **Вычислительная сложность.**  
   Одна итерация требует $O(N_d \cdot K)$ операций на документ. Для больших коллекций это может быть медленно. Существуют оптимизированные реализации (например, с использованием разреженных структур).

### Сравнение с pLSA

| Свойство | pLSA | LDA |
|----------|------|-----|
| $\theta_d$ | Параметр (фиксированный) | Случайная величина |
| $\phi_z$ | Параметр (фиксированный) | Случайная величина |
| Число параметров | $M \times K + K \times N$ | Только $\alpha, \beta$ |
| Переобучение | Возможно | Сглажено |
| Новые документы | Требует переобучения | Апостериорный вывод |
| Метод обучения | EM | Гиббс / вариационный вывод |
| Сглаживание | Нет | Да (через $\alpha, \beta$) |

### Что дальше?

Мы разобрали математические основы LDA: распределение Дирихле, порождающий процесс, совместное распределение и сэмплирование Гиббса. В следующем разделе мы применим эти формулы к конкретному корпусу из трёх документов и шаг за шагом проведём несколько итераций сэмплирования.




## 4.2.6 Численный пример LDA на учебном корпусе

### 4.2.6.1 Постановка задачи

Вернёмся к нашему корпусу из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Словарь $V$ содержит 8 слов: $N=8$. Выберем число тем $K=2$. Установим гиперпараметры $\alpha = 0.1$, $\beta = 0.01$ (типичные значения, способствующие разреженности). Для простоты будем считать, что каждое слово в каждом документе встречается ровно один раз, поэтому всего 12 слов в коллекции.

**Напоминание:** $K=2$ — это наш выбор. Модель не определяет число тем автоматически. Мы могли бы взять $K=3$, и результат был бы другим.

Исходные данные удобно записать в виде матрицы «документ–слово»:

| Документ | кошка | сидит | на | окне | собака | крыльце | спит | диване |
|----------|-------|-------|----|------|--------|---------|------|--------|
| $d_1$    | 1     | 1     | 1  | 1    | 0      | 0       | 0    | 0      |
| $d_2$    | 0     | 1     | 1  | 0    | 1      | 1       | 0    | 0      |
| $d_3$    | 1     | 0     | 1  | 0    | 0      | 0       | 1    | 1      |

Всего 12 словоупотреблений.

### 4.2.6.2 Формула сэмплирования Гиббса: тонкий момент

Напомним формулу условного распределения темы для одного слова:

$$
P(z_{d,n} = k \mid Z^{-(d,n)}, W, \alpha, \beta) \propto \frac{n_{d,k}^{-(d,n)} + \alpha_k}{\sum_{k'} \left( n_{d,k'}^{-(d,n)} + \alpha_{k'} \right)} \cdot \frac{n_{k,w}^{-(d,n)} + \beta_w}{\sum_{w'} \left( n_{k,w'}^{-(d,n)} + \beta_{w'} \right)}.
$$

При симметричных $\alpha$ и $\beta$:

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \frac{n_{k,w}^{-(d,n)} + \beta}{\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta}.
$$

**Тонкий момент:** знаменатель во втором сомножителе $\sum_{w'} n_{k,w'}^{-(d,n)} + N\beta$ **зависит от $k$**. Это общее число слов, отнесённых к теме $k$, плюс $N\beta$. Поэтому его **нельзя** просто опустить при сравнении тем, если мы хотим точные вероятности.

Однако на практике часто используют **упрощённую пропорциональность**:

$$
P(z_{d,n} = k \mid \ldots) \propto \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \left( n_{k,w}^{-(d,n)} + \beta \right),
$$

а затем нормализуют. Это допустимо, потому что знаменатель $\sum_{w'} n_{k,w'}^{-(d,n)} + N\beta$ меняется от $k$ к $k$, но при нормализации по $k$ он всё равно учитывается. В нашем крошечном примере использование полной или упрощённой формулы не меняет выбор темы, поэтому для наглядности мы будем использовать упрощённую форму, но будем помнить о её приближённости.

**Практический совет:** в реальных реализациях LDA вычисляют полную формулу:

$$
p_k = \left( n_{d,k}^{-(d,n)} + \alpha \right) \cdot \frac{n_{k,w}^{-(d,n)} + \beta}{\sum_{w'} n_{k,w'}^{-(d,n)} + N \beta},
$$

затем нормализуют $p_k$ и сэмплируют тему пропорционально $p_k$.

### 4.2.6.3 Инициализация тем

Перед началом сэмплирования необходимо присвоить каждому слову случайную начальную тему. Для воспроизводимости зададим конкретное начальное назначение:

- $d_1$: (кошка→1, сидит→1, на→2, окне→2)
- $d_2$: (собака→1, сидит→1, на→2, крыльце→2)
- $d_3$: (кошка→1, спит→1, на→2, диване→2)

Это детерминированное начальное состояние. Теперь подсчитаем счётчики $n_{d,k}$ и $n_{k,w}$.

**Счётчики $n_{d,k}$ (число слов в документе с темой k):**

| Документ | Тема 1 | Тема 2 |
|----------|--------|--------|
| $d_1$ | 2 | 2 |
| $d_2$ | 2 | 2 |
| $d_3$ | 2 | 2 |

**Счётчики $n_{k,w}$ (число вхождений слова w с темой k):**

Для темы 1:
- кошка: $d_1$ (1) + $d_3$ (1) = 2
- сидит: $d_1$ (1) + $d_2$ (1) = 2
- собака: $d_2$ (1) = 1
- спит: $d_3$ (1) = 1
- остальные (на, окне, крыльце, диване) = 0.
- **Всего:** $2+2+1+1 = 6$.

Для темы 2:
- на: $d_1$ (1) + $d_2$ (1) + $d_3$ (1) = 3
- окне: $d_1$ (1) = 1
- крыльце: $d_2$ (1) = 1
- диване: $d_3$ (1) = 1
- остальные (кошка, сидит, собака, спит) = 0.
- **Всего:** $3+1+1+1 = 6$.

Проверка: $6 + 6 = 12$ — общее число словоупотреблений.

### 4.2.6.4 Первая итерация сэмплирования

Пройдём по каждому слову в порядке документов и позиций, обновляя тему. При обновлении исключаем текущее слово из счётчиков, вычисляем вероятности для каждой темы.

**Важно:** в реальном сэмплировании Гиббса тема выбирается **случайно** пропорционально вычисленным вероятностям. В этом учебном примере для простоты и наглядности мы будем выбирать тему с максимальной вероятностью, то есть детерминированно. Это допустимо для иллюстрации, но в практических реализациях всегда используется стохастический выбор.

#### Слово 1: $d_1$, «кошка», текущая тема 1

Исключаем это слово из счётчиков:

- $n_{d_1,1}^{-(1,1)} = 2-1 = 1$
- $n_{1,\text{кошка}}^{-(1,1)} = 2-1 = 1$
- $n_1^{-(1,1)} = 6-1 = 5$ (общее число слов в теме 1 без текущего)

Вычисляем вероятности (упрощённая формула):

$$
p_1 \propto (n_{d_1,1}^{-} + \alpha) \cdot (n_{1,\text{кошка}}^{-} + \beta) = (1+0.1) \cdot (1+0.01) = 1.1 \times 1.01 = 1.111,
$$

$$
p_2 \propto (n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{кошка}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 2.1 \times 0.01 = 0.021.
$$

Нормализуем:

$$
P(z=1) = \frac{1.111}{1.111+0.021} \approx 0.981, \quad P(z=2) \approx 0.019.
$$

Выбираем тему 1 (она имеет большую вероятность). Счётчики не меняются.

#### Слово 2: $d_1$, «сидит», текущая тема 1

Исключаем:

- $n_{d_1,1}^{-} = 2-1 = 1$
- $n_{1,\text{сидит}}^{-} = 2-1 = 1$

Вычисляем:

$$
p_1 = (1+0.1) \cdot (1+0.01) = 1.111,
$$

$$
p_2 = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Выбираем тему 1. Счётчики не меняются.

#### Слово 3: $d_1$, «на», текущая тема 2

Исключаем:

- $n_{d_1,2}^{-} = 2-1 = 1$
- $n_{2,\text{на}}^{-} = 3-1 = 2$

Вычисляем:

$$
p_1 = (n_{d_1,1}^{-} + \alpha) \cdot (n_{1,\text{на}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 2.1 \times 0.01 = 0.021,
$$

$$
p_2 = (n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{на}}^{-} + \beta) = (1+0.1) \cdot (2+0.01) = 1.1 \times 2.01 = 2.211.
$$

Нормализуем:

$$
P(z=2) = \frac{2.211}{2.211+0.021} \approx 0.991.
$$

Оставляем тему 2.

#### Слово 4: $d_1$, «окне», текущая тема 2

Исключаем:

- $n_{d_1,2}^{-} = 1$ (после исключения «на» осталась 1 тема 2: «окне»)

  На самом деле, в текущем состоянии $d_1$ имеет: кошка(1), сидит(1), на(2), окне(2). Значит $n_{d_1,2}=2$. Исключаем «окне»: $n_{d_1,2}^{-}=1$.

- $n_{2,\text{окне}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (n_{d_1,1}^{-} + \alpha) \cdot (n_{1,\text{окне}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 0.021,
$$

$$
p_2 = (n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{окне}}^{-} + \beta) = (1+0.1) \cdot (0+0.01) = 0.011.
$$

Сравниваем: $0.021 > 0.011$, поэтому выбираем тему 1. Слово «окне» переходит в тему 1.

Обновляем счётчики:

- $n_{d_1,1} = 3$, $n_{d_1,2} = 1$
- $n_{1,\text{окне}} = 1$, $n_{2,\text{окне}} = 0$
- $n_1 = 6+1 = 7$, $n_2 = 6-1 = 5$

**Состояние после обработки $d_1$:**

- $d_1$: кошка(1), сидит(1), на(2), окне(1). Итого $n_{d_1,1}=3$, $n_{d_1,2}=1$.
- Счётчики $n_{k,w}$:
  - Тема 1: кошка 2, сидит 2, окне 1, собака 1, спит 1 (всего 7)
  - Тема 2: на 3, крыльце 1, диване 1 (всего 5)
- Проверка: $7+5=12$.

#### Слово 5: $d_2$, «собака», текущая тема 1

Исключаем:

- $n_{d_2,1}^{-} = 2-1 = 1$ (в $d_2$ тема 1: собака, сидит; после исключения остаётся сидит)
- $n_{1,\text{собака}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (1+0.1) \cdot (0+0.01) = 0.011,
$$

$$
p_2 = (n_{d_2,2}^{-} + \alpha) \cdot (n_{2,\text{собака}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Тема 2 имеет большую вероятность ($0.021 > 0.011$). Переводим «собака» в тему 2.

Обновляем:

- $n_{d_2,1} = 1$, $n_{d_2,2} = 3$
- $n_{1,\text{собака}} = 0$, $n_{2,\text{собака}} = 1$
- $n_1 = 6$, $n_2 = 6$

#### Слово 6: $d_2$, «сидит», текущая тема 1

Исключаем:

- $n_{d_2,1}^{-} = 1-1 = 0$ (в $d_2$ тема 1: только сидит)
- $n_{1,\text{сидит}}^{-} = 2-1 = 1$

Вычисляем:

$$
p_1 = (0+0.1) \cdot (1+0.01) = 0.101,
$$

$$
p_2 = (n_{d_2,2}^{-} + \alpha) \cdot (n_{2,\text{сидит}}^{-} + \beta) = (3+0.1) \cdot (0+0.01) = 0.031.
$$

Тема 1 имеет большую вероятность ($0.101 > 0.031$). Оставляем тему 1.

#### Слово 7: $d_2$, «на», текущая тема 2

Исключаем:

- $n_{d_2,2}^{-} = 3-1 = 2$ (собака, крыльце)
- $n_{2,\text{на}}^{-} = 3-1 = 2$

Вычисляем:

$$
p_1 = (n_{d_2,1}^{-} + \alpha) \cdot (n_{1,\text{на}}^{-} + \beta) = (1+0.1) \cdot (0+0.01) = 0.011,
$$

$$
p_2 = (2+0.1) \cdot (2+0.01) = 2.1 \times 2.01 = 4.221.
$$

Остаётся тема 2.

#### Слово 8: $d_2$, «крыльце», текущая тема 2

Исключаем:

- $n_{d_2,2}^{-} = 2$ (собака, на)
- $n_{2,\text{крыльце}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (1+0.1) \cdot (0+0.01) = 0.011,
$$

$$
p_2 = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Тема 2 имеет большую вероятность ($0.021 > 0.011$). Оставляем тему 2.

**Состояние после обработки $d_2$:**

- $d_2$: собака(2), сидит(1), на(2), крыльце(2). Итого $n_{d_2,1}=1$, $n_{d_2,2}=3$.
- Счётчики $n_{k,w}$:
  - Тема 1: кошка 2, сидит 2, окне 1, спит 1 (всего 6)
  - Тема 2: на 3, крыльце 1, диване 1, собака 1 (всего 6)
- Проверка: $6+6=12$.

#### Слово 9: $d_3$, «кошка», текущая тема 1

Исключаем:

- $n_{d_3,1}^{-} = 2-1 = 1$ (в $d_3$ тема 1: кошка, спит; после исключения остаётся спит)
- $n_{1,\text{кошка}}^{-} = 2-1 = 1$

Вычисляем:

$$
p_1 = (1+0.1) \cdot (1+0.01) = 1.111,
$$

$$
p_2 = (n_{d_3,2}^{-} + \alpha) \cdot (n_{2,\text{кошка}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Оставляем тему 1.

#### Слово 10: $d_3$, «спит», текущая тема 1

Исключаем:

- $n_{d_3,1}^{-} = 1-1 = 0$ (в $d_3$ тема 1: только кошка)
- $n_{1,\text{спит}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (0+0.1) \cdot (0+0.01) = 0.001,
$$

$$
p_2 = (n_{d_3,2}^{-} + \alpha) \cdot (n_{2,\text{спит}}^{-} + \beta) = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Тема 2 имеет большую вероятность. Переводим «спит» в тему 2.

Обновляем:

- $n_{d_3,1} = 1$, $n_{d_3,2} = 3$
- $n_{1,\text{спит}} = 0$, $n_{2,\text{спит}} = 1$
- $n_1 = 5$, $n_2 = 7$

#### Слово 11: $d_3$, «на», текущая тема 2

Исключаем:

- $n_{d_3,2}^{-} = 3-1 = 2$ (спит, диване)
- $n_{2,\text{на}}^{-} = 3-1 = 2$

Вычисляем:

$$
p_1 = (n_{d_3,1}^{-} + \alpha) \cdot (n_{1,\text{на}}^{-} + \beta) = (1+0.1) \cdot (0+0.01) = 0.011,
$$

$$
p_2 = (2+0.1) \cdot (2+0.01) = 4.221.
$$

Остаётся тема 2.

#### Слово 12: $d_3$, «диване», текущая тема 2

Исключаем:

- $n_{d_3,2}^{-} = 2$ (спит, на)
- $n_{2,\text{диване}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (1+0.1) \cdot (0+0.01) = 0.011,
$$

$$
p_2 = (2+0.1) \cdot (0+0.01) = 0.021.
$$

Тема 2 более вероятна, оставляем.

**Состояние после первой полной итерации:**

**Назначения тем:**

- $d_1$: кошка(1), сидит(1), на(2), окне(1) → $n_{d_1,1}=3$, $n_{d_1,2}=1$
- $d_2$: собака(2), сидит(1), на(2), крыльце(2) → $n_{d_2,1}=1$, $n_{d_2,2}=3$
- $d_3$: кошка(1), спит(2), на(2), диване(2) → $n_{d_3,1}=1$, $n_{d_3,2}=3$

**Счётчики $n_{k,w}$:**

- Тема 1: кошка 2, сидит 2, окне 1 (всего 5)
- Тема 2: на 3, крыльце 1, диване 1, собака 1, спит 1 (всего 7)
- Проверка: $5+7=12$.

### 4.2.6.5 Вторая итерация (кратко)

На второй итерации повторяем проход по всем словам с использованием новых счётчиков. Покажем несколько примеров.

#### Слово 1: $d_1$, «кошка», тема 1

Исключаем:

- $n_{d_1,1}^{-} = 3-1 = 2$
- $n_{1,\text{кошка}}^{-} = 2-1 = 1$

Вычисляем:

$$
p_1 = (2+0.1) \cdot (1+0.01) = 2.121,
$$

$$
p_2 = (n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{кошка}}^{-} + \beta) = (1+0.1) \cdot (0+0.01) = 0.011.
$$

Вероятность темы 1 почти 1. Оставляем.

#### Слово 4: $d_1$, «окне», тема 1

Исключаем:

- $n_{d_1,1}^{-} = 2$ (кошка, сидит)
- $n_{1,\text{окне}}^{-} = 1-1 = 0$

Вычисляем:

$$
p_1 = (2+0.1) \cdot (0+0.01) = 0.021,
$$

$$
p_2 = (n_{d_1,2}^{-} + \alpha) \cdot (n_{2,\text{окне}}^{-} + \beta) = (1+0.1) \cdot (0+0.01) = 0.011.
$$

Тема 1 чуть более вероятна ($0.021 > 0.011$). Оставляем тему 1.

Остальные слова, вероятно, сохранят свои темы, так как структура уже стабилизировалась. В нашем учебном примере из-за крошечного размера корпуса назначения тем действительно быстро перестают меняться. В реальных задачах для стабилизации цепи обычно требуется значительно больше итераций (например, сотни или тысячи), и процесс сходимости отслеживают по стабилизации логарифма правдоподобия или другим диагностикам.

### 4.2.6.6 Сходимость

После нескольких итераций состояние сходится к следующему (округлённо):

**Итоговое назначение тем:**

- $d_1$: кошка(1), сидит(1), на(2), окне(1) (тема 1: 3, тема 2: 1)
- $d_2$: собака(2), сидит(1), на(2), крыльце(2) (тема 1: 1, тема 2: 3)
- $d_3$: кошка(1), спит(2), на(2), диване(2) (тема 1: 1, тема 2: 3)

**Итоговые счётчики $n_{k,w}$:**

- Тема 1: кошка 2, сидит 2, окне 1 (всего 5)
- Тема 2: на 3, крыльце 1, диване 1, собака 1, спит 1 (всего 7)

Это состояние совпадает с концом первой итерации, что говорит о быстрой сходимости для такого маленького корпуса.

---

## 4.2.7 Оценка параметров $\theta$ и $\phi$

После сходимости мы оцениваем параметры модели по формулам:

$$
\hat{\theta}_{d,k} = \frac{n_{d,k} + \alpha}{\sum_{k'} (n_{d,k'} + \alpha)},
$$

$$
\hat{\phi}_{k,w} = \frac{n_{k,w} + \beta}{\sum_{w'} (n_{k,w'} + \beta)}.
$$

### 4.2.7.1 Оценка $\theta_{d,k}$

**Для $d_1$:** $n_{d_1,1}=3$, $n_{d_1,2}=1$, $\alpha=0.1$.

$$
\theta_{d_1,1} = \frac{3 + 0.1}{4 + 2 \times 0.1} = \frac{3.1}{4.2} \approx 0.738,
$$

$$
\theta_{d_1,2} = \frac{1 + 0.1}{4.2} = \frac{1.1}{4.2} \approx 0.262.
$$

**Для $d_2$:** $n_{d_2,1}=1$, $n_{d_2,2}=3$.

$$
\theta_{d_2,1} = \frac{1.1}{4.2} \approx 0.262, \quad \theta_{d_2,2} = \frac{3.1}{4.2} \approx 0.738.
$$

**Для $d_3$:** аналогично $d_2$, так как счётчики такие же.

$$
\theta_{d_3,1} \approx 0.262, \quad \theta_{d_3,2} \approx 0.738.
$$

**Сводная таблица $\theta$:**

| Документ | $\theta_{d,1}$ | $\theta_{d,2}$ |
|----------|-----------------|-----------------|
| $d_1$ | 0.738 | 0.262 |
| $d_2$ | 0.262 | 0.738 |
| $d_3$ | 0.262 | 0.738 |

**Проверка:** для каждого документа сумма $\theta_{d,1} + \theta_{d,2} = 1$. Например, $0.738 + 0.262 = 1.000$.

### 4.2.7.2 Оценка $\phi_{k,w}$

**Для темы 1:** сумма $n_{1,w'} = 5$, плюс $8 \times 0.01 = 0.08$. Знаменатель $5 + 0.08 = 5.08$.

| слово | $n_{1,w}$ | $\hat{\phi}_{1,w} = (n_{1,w}+0.01)/5.08$ |
|-------|-----------|------------------------------------------|
| кошка | 2 | $2.01/5.08 \approx 0.396$ |
| сидит | 2 | $2.01/5.08 \approx 0.396$ |
| окне  | 1 | $1.01/5.08 \approx 0.199$ |
| собака | 0 | $0.01/5.08 \approx 0.002$ |
| на    | 0 | $0.01/5.08 \approx 0.002$ |
| крыльце | 0 | $0.01/5.08 \approx 0.002$ |
| спит  | 0 | $0.01/5.08 \approx 0.002$ |
| диване | 0 | $0.01/5.08 \approx 0.002$ |

**Проверка суммы:**

$$
0.396 + 0.396 + 0.199 + 5 \times 0.002 = 0.991 + 0.010 = 1.001 \approx 1.
$$

**Для темы 2:** сумма $n_{2,w'} = 7$, знаменатель $7 + 0.08 = 7.08$.

| слово | $n_{2,w}$ | $\hat{\phi}_{2,w} = (n_{2,w}+0.01)/7.08$ |
|-------|-----------|------------------------------------------|
| на    | 3 | $3.01/7.08 \approx 0.425$ |
| крыльце | 1 | $1.01/7.08 \approx 0.143$ |
| диване | 1 | $1.01/7.08 \approx 0.143$ |
| собака | 1 | $1.01/7.08 \approx 0.143$ |
| спит  | 1 | $1.01/7.08 \approx 0.143$ |
| кошка | 0 | $0.01/7.08 \approx 0.001$ |
| сидит | 0 | $0.01/7.08 \approx 0.001$ |
| окне  | 0 | $0.01/7.08 \approx 0.001$ |

**Проверка суммы:**

$$
0.425 + 4 \times 0.143 + 3 \times 0.001 = 0.425 + 0.572 + 0.003 = 1.000.
$$

### 4.2.7.3 Итоговая матрица $\phi$

Сведём всё в одну таблицу:

| слово | $\phi_{1,w}$ (тема 1) | $\phi_{2,w}$ (тема 2) |
|-------|----------------------|----------------------|
| кошка | 0.396 | 0.001 |
| сидит | 0.396 | 0.001 |
| на    | 0.002 | 0.425 |
| окне  | 0.199 | 0.001 |
| собака | 0.002 | 0.143 |
| крыльце | 0.002 | 0.143 |
| спит  | 0.002 | 0.143 |
| диване | 0.002 | 0.143 |

### 4.2.7.4 Итоговая матрица $P(w \mid d)$

По обученной модели можно вычислить вероятность каждого слова в каждом документе:

$$
P(w \mid d) = \sum_{k=1}^{K} \phi_{k,w} \cdot \theta_{d,k}.
$$

Приведём матрицу $P(w \mid d)$ для всех документов (округлённо до 4 знаков).

| Слово | $P(w \mid d_1)$ | $P(w \mid d_2)$ | $P(w \mid d_3)$ |
|-------|-----------------|-----------------|-----------------|
| кошка | $0.396 \cdot 0.738 + 0.001 \cdot 0.262 = 0.2925$ | $0.396 \cdot 0.262 + 0.001 \cdot 0.738 = 0.1045$ | $0.396 \cdot 0.262 + 0.001 \cdot 0.738 = 0.1045$ |
| сидит | $0.396 \cdot 0.738 + 0.001 \cdot 0.262 = 0.2925$ | $0.396 \cdot 0.262 + 0.001 \cdot 0.738 = 0.1045$ | $0.396 \cdot 0.262 + 0.001 \cdot 0.738 = 0.1045$ |
| на    | $0.002 \cdot 0.738 + 0.425 \cdot 0.262 = 0.1128$ | $0.002 \cdot 0.262 + 0.425 \cdot 0.738 = 0.3142$ | $0.002 \cdot 0.262 + 0.425 \cdot 0.738 = 0.3142$ |
| окне  | $0.199 \cdot 0.738 + 0.001 \cdot 0.262 = 0.1471$ | $0.199 \cdot 0.262 + 0.001 \cdot 0.738 = 0.0529$ | $0.199 \cdot 0.262 + 0.001 \cdot 0.738 = 0.0529$ |
| собака | $0.002 \cdot 0.738 + 0.143 \cdot 0.262 = 0.0390$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ |
| крыльце | $0.002 \cdot 0.738 + 0.143 \cdot 0.262 = 0.0390$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ |
| спит  | $0.002 \cdot 0.738 + 0.143 \cdot 0.262 = 0.0390$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ |
| диване | $0.002 \cdot 0.738 + 0.143 \cdot 0.262 = 0.0390$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ | $0.002 \cdot 0.262 + 0.143 \cdot 0.738 = 0.1060$ |

**Проверка:** сумма по каждому столбцу должна быть равна 1. Проверим для $d_1$:

$$
0.2925 + 0.2925 + 0.1128 + 0.1471 + 0.0390 + 0.0390 + 0.0390 + 0.0390 = 1.0009 \approx 1.
$$

Аналогично для $d_2$ и $d_3$.

**Как читать эту матрицу?** Она показывает, с какой вероятностью модель ожидает увидеть каждое слово в каждом документе. Например, в $d_1$ наиболее вероятны слова «кошка» (0.2925) и «сидит» (0.2925), затем «окне» (0.1471) и «на» (0.1128). Это отражает то, что тема 1, доминирующая в $d_1$ (0.738), включает «кошку», «сидит», «окне», а тема 2 добавляет «на». В $d_2$ максимальна вероятность слова «на» (0.3142), затем «собака», «крыльце», «спит», «диване» (по 0.1060), что связано с доминированием темы 2 (0.738).

---

## 4.2.8 Интерпретация и сравнение с pLSA

### 4.2.8.1 Интерпретация тем

**Тема 1** сконцентрирована на словах:

| слово | $\phi_{1,w}$ |
|-------|-------------|
| кошка | 0.396 |
| сидит | 0.396 |
| окне  | 0.199 |
| остальные | 0.002 |

Это соответствует контексту «кошка сидит на окне» (документ 1).

**Тема 2** выделяет:

| слово | $\phi_{2,w}$ |
|-------|-------------|
| на | 0.425 |
| крыльце | 0.143 |
| диване | 0.143 |
| собака | 0.143 |
| спит | 0.143 |
| остальные | 0.001 |

Это более смешанная тема, включающая предлог «на» и слова из документов 2 и 3.

### 4.2.8.2 Сравнение с pLSA

В pLSA темы получились более сбалансированными:

- Тема 1: животные и действия («кошка» 0.258, «собака» 0.240, «сидит» 0.207, «спит» 0.156).
- Тема 2: места и предлоги («на» 0.421, «крыльце» 0.177, «диване» 0.171, «окне» 0.156).

В LDA из-за малого размера корпуса и конкретной инициализации темы получились более смешанными:

- Тема 1 сфокусировалась на связке «кошка сидит на окне» (документ 1).
- Тема 2 — на остальных словах, включая «собака» и «спит», которые в pLSA относились к теме 1.

Это объясняется тем, что LDA — вероятностная модель с априорными ограничениями, и на крошечных данных результат может зависеть от начального состояния и случайности сэмплирования.

### 4.2.8.3 Преимущества LDA

Ключевое преимущество LDA перед pLSA видно в оценках $\theta_d$: благодаря добавлению $\alpha$ и $\beta$ параметры сглажены, и ни одно значение не обращается в нуль, даже для слов, не встретившихся в теме.

Например, слово «собака» в теме 1 имеет ненулевую вероятность ($0.002$), хотя в обучающих данных не встречалось с темой 1. Это обеспечивает устойчивость к переобучению и позволяет модели обобщать на новые документы. В pLSA без сглаживания вероятности для не встреченных комбинаций могли бы быть нулевыми.

Кроме того, LDA способна обрабатывать новые документы: зафиксировав обученные $\phi_z$, можно вывести распределение тем $\theta_{\text{new}}$ для нового текста с помощью нескольких дополнительных итераций сэмплирования или вариационного вывода. В pLSA для этого требовалось бы переобучение параметров документа с нуля, но глобальные темы были бы фиксированы, что делало процесс не вполне байесовским.

---

## 4.2.9 Заключение по LDA

Латентное размещение Дирихле стало стандартом тематического моделирования благодаря своей статистической строгости, способности к обобщению и естественной интерпретируемости. Она устраняет недостатки pLSA, вводя байесовские априорные распределения, которые сглаживают оценки и позволяют корректно работать с новыми текстами. В практических приложениях LDA применяется для анализа больших коллекций документов, выявления скрытых тем, кластеризации и снижения размерности.

Мы рассмотрели вывод с помощью сэмплирования Гиббса, который интуитивно понятен и широко используется в реализациях. Существуют и другие методы, такие как вариационный вывод и онлайн-варианты, обеспечивающие масштабируемость на огромные корпуса. Понимание математических основ LDA необходимо для осознанного применения модели и её расширений, таких как динамические тематические модели, коррелированные тематические модели и нейротематические подходы.

На этом завершается обзор классических методов векторизации текста — от one-hot encoding до LDA. Каждый из этих подходов вносил вклад в решение проблемы числового представления естественного языка, постепенно приближая нас к современным нейросетевым эмбеддингам, которые будут рассмотрены в следующих лекциях.
